# Visualizing Relationship Between μ Structure and Slip Inversion

This notebook visualizes:
1. μ structure on fault surface (from 3D model)
2. Slip comparison: 3D inversion vs. homogeneous inversion
3. Displacement difference at GNSS stations
4. μ contrast across fault

**Case:** nicoyaCK4, 2-stripe pattern, DeShon3D_ref_4 vs mul40u40

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import cm
import pyvista as pv
import pygmt

# FEniCS for mesh loading
import dolfin as dl

print("Libraries imported successfully!")

## 1. Setup and Configuration

In [ ]:
# Paths
datadir = "/home/staff/chao/SSEinv/Nicoya/data/"
meshpath = "/home/staff/chao/SSEinv/Nicoya/mesh/"
resultpath = "/home/staff/chao/SSEinv/Nicoya/syn_slip/"

# PLATETYPE = "CK"
PLATETYPE = "SLAB2"

# Mesh and slip pattern
### flatter fault surface
# meshname = "nicoya3"    # flatter fault surface
meshname = "nicoyaden_sm"    # flatter fault surface

slip_str_gt = "_stripe_x0_y-40_lx90_dx35_rot0_ms0.0785_pou"

# Structure strings
het3d_str = "_DeShon3D_ref_4"  # 3D heterogeneous model
# het3d_str = "_DeShon3D_ref_1"  # 3D heterogeneous model
homo_str = "_mul40u40"          # Homogeneous reference

# Inversion string
inv_str = "_synlockbd_w945390_gs1e+02_ds1e-07"

# Fault boundary ID
fault_id = 7

# Subdomain IDs for Δμ calculation
blockleft = 8   # subducting plate (footwall)
blockright = 9  # overriding plate (hanging wall)

# Problem type for slip loading
# For locking problems: slip files are in meters, normalize by amp to get coupling ratio
problem_type = 'locking'  # 'coseismic' or 'locking'
V_norm = 78.5 / 1e3  # Trench-normal long-term loading: 78.5 mm/yr = 0.0785 m/yr
amp = V_norm  # Normalization amplitude (for locking: max coupling = 1 = complete locking)

print(f"Configuration:")
print(f"  Mesh: {meshname}")
print(f"  Slip pattern: {slip_str_gt}")
print(f"  3D structure: {het3d_str}")
print(f"  Homogeneous: {homo_str}")
print(f"  Problem type: {problem_type}")
print(f"  Normalization amp: {amp:.4f} m/yr (V_norm = {V_norm*1e3:.1f} mm/yr)")

## 2. Load Mesh and Extract Fault Facets

**Note:** We use fault **facets** (triangular faces) rather than vertices throughout this analysis because:
- Normals are naturally defined on facets
- μ contrast calculation requires facet centers and normals
- Provides ~2× more sampling points than vertices (~6185 facets vs ~3165 vertices)

In [ ]:
# Load mesh
mesh = dl.Mesh(meshpath + meshname + '.xml')
boundaries = dl.MeshFunction("size_t", mesh, meshpath + meshname + '_facet_region.xml')
subdomains = dl.MeshFunction("size_t", mesh, meshpath + meshname + '_physical_region.xml')

print(f"Mesh loaded: {mesh.num_vertices()} vertices, {mesh.num_cells()} cells")
print(f"Mesh dimension: {mesh.topology().dim()}")

In [ ]:
# Load fault geometry (same coordinates used for slip data)
fault_geom_file = resultpath + f"fault_geometry_{meshname}.txt"
fault_geom = pd.read_csv(fault_geom_file, sep=r'\s+', skiprows=lambda x: x % 3 != 1,
                        names=['x', 'y', 'z'])
fault_coords = fault_geom[['x', 'y', 'z']].values

print(f"Loaded fault coordinates: {len(fault_coords)} nodes (matching slip data)")

# Compute normals at fault vertices by averaging nearby facet normals
def compute_vertex_normals(mesh, boundaries, fault_id, vertex_coords):
    """Compute normals at fault vertices by averaging surrounding facet normals"""
    from scipy.spatial import cKDTree

    # First get all facet centers and normals
    facet_centers = []
    facet_normals_list = []

    for facet in dl.facets(mesh):
        if boundaries[facet] == fault_id:
            vertices = facet.entities(0)
            coords = mesh.coordinates()[vertices]
            center = coords.mean(axis=0)

            v1 = coords[1] - coords[0]
            v2 = coords[2] - coords[0]
            normal = np.cross(v1, v2)
            normal = normal / np.linalg.norm(normal)

            facet_centers.append(center)
            facet_normals_list.append(normal)

    facet_centers = np.array(facet_centers)
    facet_normals_array = np.array(facet_normals_list)

    # Build KDTree for nearest neighbor lookup
    tree = cKDTree(facet_centers)

    # For each vertex, average normals from 5 nearest facets
    vertex_normals = []
    for vc in vertex_coords:
        _, indices = tree.query(vc, k=5)
        avg_normal = facet_normals_array[indices].mean(axis=0)
        avg_normal = avg_normal / np.linalg.norm(avg_normal)
        vertex_normals.append(avg_normal)

    return np.array(vertex_normals)

print("Computing normals at fault vertices...")

fault_normals = compute_vertex_normals(mesh, boundaries, fault_id, fault_coords)

# Determine and correct normal direction
def determine_normal_direction(mesh, subdomains, coords, normals, blockleft_id, blockright_id):
    """Ensure normals point from FW (blockleft) to HW (blockright)"""
    n_test = min(10, len(coords))
    flip_count = 0

    for i in range(n_test):
        center = coords[i]
        normal = normals[i]
        offset_test = 100.0
        point_plus = center + offset_test * normal
        point_minus = center - offset_test * normal

        cell_plus = mesh.bounding_box_tree().compute_first_entity_collision(dl.Point(point_plus))
        cell_minus = mesh.bounding_box_tree().compute_first_entity_collision(dl.Point(point_minus))

        if cell_plus < mesh.num_cells() and cell_minus < mesh.num_cells():
            subdomain_plus = subdomains[int(cell_plus)]
            subdomain_minus = subdomains[int(cell_minus)]

            # Normal should point from FW (blockleft) to HW (blockright)
            # So: normal direction should have blockright, -normal should have blockleft
            if subdomain_minus == blockright_id and subdomain_plus == blockleft_id:
                flip_count += 1

    should_flip = flip_count > n_test // 2

    if should_flip:
        print(f"Flipping normal direction to point from FW to HW")
        return -normals
    else:
        print(f"Normal direction already correct (FW→HW)")
        return normals.copy()

fault_normals = determine_normal_direction(mesh, subdomains, fault_coords, fault_normals,
blockleft, blockright)


def spatial_median_filter(coords, values, k_neighbors=10):
    """Apply spatial median filter using k nearest neighbors"""
    from scipy.spatial import cKDTree
    
    tree = cKDTree(coords)
    filtered = np.zeros_like(values)

    for i in range(len(values)):
        if np.isnan(values[i]):
            filtered[i] = np.nan
            continue

        # Find k nearest neighbors
        distances, indices = tree.query(coords[i], k=k_neighbors+1)  # +1 includes self
        neighbor_values = values[indices[1:]]  # Exclude self

        # Remove NaN neighbors
        neighbor_values = neighbor_values[~np.isnan(neighbor_values)]

        if len(neighbor_values) > 0:
            # Use median of neighbors (robust to outliers)
            filtered[i] = np.median(neighbor_values)
        else:
            filtered[i] = values[i]

    return filtered

# Sample mu function with layer-based averaging - TWO METHODS
def sample_mu_across_fault_layered(mu_grid, coords, normals, layer_center, layer_thickness,
                                    n_samples=5, field_name='shear modulus', 
                                    method='linear'):
    """
    Sample μ on both sides of fault by averaging over a thin layer

    Parameters:
    -----------
    mu_grid : PyVista UnstructuredGrid
        Grid containing shear modulus field
    coords : ndarray (N, 3)
        Fault vertex coordinates
    normals : ndarray (N, 3)
        Fault vertex normals (pointing from FW to HW)
    layer_center : float
        Distance from fault to center of sampling layer (meters)
    layer_thickness : float
        Thickness of sampling layer (meters)
    n_samples : int
        Number of sampling points within each layer
    field_name : str
        Name of shear modulus field in mu_grid
    method : str
        'linear' - scipy linear interpolation (smooth, artificial)
        'mesh_vertices' - use actual mesh vertices (true FEniCS values)

    Returns:
    --------
    mu_hw : ndarray
        Layer-averaged μ on hanging wall side (GPa)
    mu_fw : ndarray
        Layer-averaged μ on footwall side (GPa)
    """
    from scipy.interpolate import griddata
    from scipy.spatial import cKDTree

    # Create offsets within the layer
    offset_min = layer_center - layer_thickness / 2.0
    offset_max = layer_center + layer_thickness / 2.0

    print(f"\nLayer-based μ sampling:")
    print(f"  Layer center: {layer_center/1e3:.1f} km from fault")
    print(f"  Layer thickness: {layer_thickness/1e3:.1f} km")
    print(f"  Sampling range: [{offset_min/1e3:.1f}, {offset_max/1e3:.1f}] km")
    print(f"  Number of samples per layer: {n_samples}")
    print(f"  Method: {method}")

    grid_points = mu_grid.points
    mu_values = mu_grid.point_data[field_name]

    if method == 'mesh_vertices':
        print(f"  Using ACTUAL MESH VERTICES with subdomain verification")
        print(f"  Sampling strictly along normal direction")

        # Build KDTree for mesh points
        tree = cKDTree(grid_points)

        # Get subdomain IDs for each mesh point
        # We need to query which subdomain each mesh vertex belongs to
        print(f"  Extracting subdomain IDs for mesh vertices...")

        # Create mapping from mesh coordinates to subdomain
        mesh_coords = mesh.coordinates()
        mesh_to_subdomain = {}

        for cell_idx in range(mesh.num_cells()):
            subdomain_id = subdomains[cell_idx]
            cell = dl.Cell(mesh, cell_idx)
            for vertex in dl.vertices(cell):
                vertex_idx = vertex.index()
                mesh_to_subdomain[vertex_idx] = subdomain_id

        # Now match mu_grid points to mesh vertices
        tree_mesh = cKDTree(mesh_coords)
        grid_to_subdomain = np.zeros(len(grid_points), dtype=int)

        for i, pt in enumerate(grid_points):
            dist, idx = tree_mesh.query(pt, k=1)
            if dist < 100:  # Within 100m of a mesh vertex
                if idx in mesh_to_subdomain:
                    grid_to_subdomain[i] = mesh_to_subdomain[idx]

        print(f"  Mapped {np.sum(grid_to_subdomain > 0)} grid points to subdomains")

        # Define sampling offsets
        offsets = np.linspace(offset_min, offset_max, n_samples)

        # Larger search radius but verify subdomain
        max_search_dist = 50000.0  # 50 km

        mu_hw_list = []
        mu_fw_list = []

        n_missing_hw = 0
        n_missing_fw = 0

        for i, (coord, normal) in enumerate(zip(coords, normals)):
            hw_values = []
            fw_values = []

            for offset in offsets:
                point_hw = coord + offset * normal
                point_fw = coord - offset * normal

                # Find k nearest neighbors instead of just 1
                dists_hw, indices_hw = tree.query(point_hw, k=20)
                dists_fw, indices_fw = tree.query(point_fw, k=20)

                # HW side: Find nearest point from HW subdomain (blockright)
                for dist, idx in zip(dists_hw, indices_hw):
                    if dist < max_search_dist and grid_to_subdomain[idx] == blockright:
                        hw_values.append(mu_values[idx])
                        break  # Use first valid point

                # FW side: Find nearest point from FW subdomain (blockleft)
                for dist, idx in zip(dists_fw, indices_fw):
                    if dist < max_search_dist and grid_to_subdomain[idx] == blockleft:
                        fw_values.append(mu_values[idx])
                        break  # Use first valid point

            if len(hw_values) > 0:
                mu_hw_list.append(np.mean(hw_values))
            else:
                mu_hw_list.append(np.nan)
                n_missing_hw += 1

            if len(fw_values) > 0:
                mu_fw_list.append(np.mean(fw_values))
            else:
                mu_fw_list.append(np.nan)
                n_missing_fw += 1

        mu_hw = np.array(mu_hw_list)
        mu_fw = np.array(mu_fw_list)

        # After sampling with robust_average:
        print("\nApplying spatial consistency check...")
        mu_hw = spatial_median_filter(fault_coords, mu_hw, k_neighbors=10)
        mu_fw = spatial_median_filter(fault_coords, mu_fw, k_neighbors=10)

        # 3. Fill any remaining NaN values
        print(f"\nFilling NaN values...")
        print(f"  NaN count before: HW={np.sum(np.isnan(mu_hw))}, FW={np.sum(np.isnan(mu_fw))}")

        valid_hw = ~np.isnan(mu_hw)
        if np.sum(valid_hw) > 3:
            mu_hw = griddata(fault_coords[valid_hw], mu_hw[valid_hw],
                            fault_coords, method='nearest')

        valid_fw = ~np.isnan(mu_fw)
        if np.sum(valid_fw) > 3:
            mu_fw = griddata(fault_coords[valid_fw], mu_fw[valid_fw],
                            fault_coords, method='nearest')
        print(f"  NaN count after: HW={np.sum(np.isnan(mu_hw))}, FW={np.sum(np.isnan(mu_fw))}")
    
    else:
        # METHOD: Linear interpolation (original approach)
        print(f"  Interpolation: linear (smooth interpolation between mesh points)")

        offsets = np.linspace(offset_min, offset_max, n_samples)

        mu_hw_samples = []
        mu_fw_samples = []

        for offset in offsets:
            # Points on hanging wall side (positive normal direction)
            points_hw = coords + offset * normals
            # Points on footwall side (negative normal direction)
            points_fw = coords - offset * normals

            # Interpolate μ at these points using LINEAR interpolation
            mu_hw_at_offset = griddata(grid_points, mu_values, points_hw, method='linear')
            mu_fw_at_offset = griddata(grid_points, mu_values, points_fw, method='linear')

            mu_hw_samples.append(mu_hw_at_offset)
            mu_fw_samples.append(mu_fw_at_offset)

        # Average over all samples in the layer
        mu_hw = np.nanmean(mu_hw_samples, axis=0)
        mu_fw = np.nanmean(mu_fw_samples, axis=0)

    print(f"\nLayer-averaged μ (HW): min={np.nanmin(mu_hw):.1f}, max={np.nanmax(mu_hw):.1f}, mean={np.nanmean(mu_hw):.1f} GPa")
    print(f"Layer-averaged μ (FW): min={np.nanmin(mu_fw):.1f}, max={np.nanmax(mu_fw):.1f}, mean={np.nanmean(mu_fw):.1f} GPa")

    return mu_hw, mu_fw


## 3. Load μ from XDMF Files

In [ ]:
# Function to load xdmf as pyvista grid
def load_xdmf_as_pyvista(filepath):
    """
    Load XDMF file and convert to PyVista UnstructuredGrid
    """
    import meshio
    
    # Read with meshio
    mesh_data = meshio.read(filepath)
    
    # Get points
    points = mesh_data.points
    
    # Get cells - assume tetrahedral mesh
    cells_data = []
    for cell_block in mesh_data.cells:
        cell_type = cell_block.type
        cell_data = cell_block.data
        
        # Map meshio cell type to pyvista
        if cell_type == 'tetra':
            pv_cell_type = pv.CellType.TETRA
            n_points_per_cell = 4
        elif cell_type == 'hexahedron':
            pv_cell_type = pv.CellType.HEXAHEDRON  
            n_points_per_cell = 8
        else:
            continue
            
        # Format for pyvista: [n_points, p0, p1, ..., n_points, p0, p1, ...]
        for cell in cell_data:
            cells_data.append(n_points_per_cell)
            cells_data.extend(cell)
    
    cells = np.array(cells_data)
    
    # Create cell types array
    n_cells = len(cells_data) // (n_points_per_cell + 1)
    cell_types = np.full(n_cells, pv_cell_type, dtype=np.uint8)
    
    # Create unstructured grid
    grid = pv.UnstructuredGrid(cells, cell_types, points)
    
    # Add point/cell data
    for name, data in mesh_data.point_data.items():
        grid.point_data[name] = data
    
    for name, data in mesh_data.cell_data.items():
        # Cell data comes as list of arrays (one per cell block)
        if isinstance(data, list) and len(data) > 0:
            grid.cell_data[name] = data[0]
        else:
            grid.cell_data[name] = data
    
    return grid

print("Load function defined")

In [ ]:
# Load mu grids
mu_3d_file = resultpath + f"mu_true_{meshname}{het3d_str}.xdmf"
mu_hom_file = resultpath + f"mu_true_{meshname}{homo_str}.xdmf"

print(f"Loading 3D mu from: {mu_3d_file}")
mu_3d_grid = load_xdmf_as_pyvista(mu_3d_file)

print(f"Loading homogeneous mu from: {mu_hom_file}")
mu_hom_grid = load_xdmf_as_pyvista(mu_hom_file)

# Check field names
print(f"\n3D mu fields: {list(mu_3d_grid.point_data.keys())}")
print(f"Homogeneous mu fields: {list(mu_hom_grid.point_data.keys())}")

# Print basic statistics for 3D heterogeneous model
print(f"\n3D Heterogeneous Model Statistics:")
print(f"  Mean: {mu_3d_grid['shear modulus'].mean():.2f} GPa")
print(f"  Min:  {mu_3d_grid['shear modulus'].min():.2f} GPa")
print(f"  Max:  {mu_3d_grid['shear modulus'].max():.2f} GPa")
print(f"  Std:  {mu_3d_grid['shear modulus'].std():.2f} GPa")

# Print basic statistics for homogeneous model
print(f"\nHomogeneous Model Statistics:")
print(f"  Mean: {mu_hom_grid['shear modulus'].mean():.2f} GPa")
print(f"  Min:  {mu_hom_grid['shear modulus'].min():.2f} GPa")
print(f"  Max:  {mu_hom_grid['shear modulus'].max():.2f} GPa")
print(f"  Std:  {mu_hom_grid['shear modulus'].std():.2f} GPa")

# Compute anomaly statistics
mu_3d_grid_mean = mu_3d_grid['shear modulus'].mean()
mu_hom_grid_mean = mu_hom_grid['shear modulus'].mean()

# reference mu for anomaly calculations
# mu_ref = mu_hom_grid_mean

if het3d_str == "_DeShon3D_ref_4":
    # mu_ref = 53.20    # this is mean of the orginal model, not amplified
    mu_ref = mu_3d_grid_mean
    # mu_ref = (mu_3d_grid['shear modulus'].min()+mu_3d_grid['shear modulus'].max())/2

elif het3d_str == "_mul55u25":
    # mu_ref = (mu_3d_grid['shear modulus'].min()+mu_3d_grid['shear modulus'].max())/2
    mu_ref = 40.0

print(f"\nUsing reference μ = {mu_ref:.2f} GPa for anomaly calculations")

mu_anomaly = (mu_3d_grid['shear modulus'] - mu_ref) / mu_ref * 100
print(f"\nAnomaly Statistics (%):")
print(f"  Mean: {mu_anomaly.mean():.2f}%")
print(f"  Min:  {mu_anomaly.min():.2f}%")
print(f"  Max:  {mu_anomaly.max():.2f}%")
print(f"  Std:  {mu_anomaly.std():.2f}%")

In [ ]:
# Extract mu at fault coordinates 
from scipy.interpolate import griddata

# Extract grid points and mu values
grid_points = mu_3d_grid.points
mu_3d_values = mu_3d_grid.point_data['shear modulus']
mu_hom_values = mu_hom_grid.point_data['shear modulus']

# Interpolate to fault coordinates
print("\n3D mu at fault:")
mu_fault_3d = griddata(grid_points, mu_3d_values, fault_coords, method='nearest')
print(f"μ: min={mu_fault_3d.min():.1f}, max={mu_fault_3d.max():.1f}, mean={mu_fault_3d.mean():.1f} GPa, n={len(mu_fault_3d)}")

print("\nHomogeneous mu at fault:")
mu_fault_hom = griddata(grid_points, mu_hom_values, fault_coords, method='nearest')
print(f"μ: min={mu_fault_hom.min():.1f}, max={mu_fault_hom.max():.1f} GPa, n={len(mu_fault_hom)}")

mu_anomaly = (mu_fault_3d - mu_ref) / mu_ref * 100
print(f"\nμ anomaly: min={mu_anomaly.min():.1f}%, max={mu_anomaly.max():.1f}%, n={len(mu_anomaly)}")


## 4. Load Slip Data

Slip files contain only (s_strike, s_dip) columns. Coordinates come from fault_geometry file.

**For locking problems:**
- Raw slip files are stored in meters (physical units from forward/inverse modeling)
- Must normalize by `amp` (trench-normal loading rate) to get **coupling ratio**
- Coupling ratio: 0 = free slip, 1 = complete locking (plate velocity)
- This normalization is applied to ALL slip files (ground truth AND inferred)

In [ ]:
# Load fault geometry (coordinates)

print(f"Fault geometry loaded: {len(fault_geom)} nodes")
print(f"  x: [{fault_geom['x'].min()/1e3:.3f}, {fault_geom['x'].max()/1e3:.3f}] km")
print(f"  y: [{fault_geom['y'].min()/1e3:.3f}, {fault_geom['y'].max()/1e3:.3f}] km")
print(f"  z: [{fault_geom['z'].min()/1e3:.3f}, {fault_geom['z'].max()/1e3:.3f}] km")


In [ ]:
# Function to read slip files
def read_slip_file(filename, fault_geom, is_ground_truth=False, problem_type='locking', amp=None):
    """
    Read slip file containing (s_strike, s_dip) and combine with fault geometry
    
    Parameters:
    -----------
    filename : str
        Path to slip file
    fault_geom : DataFrame
        Fault geometry with (x, y, z) coordinates
    is_ground_truth : bool
        If True, mark as ground truth (affects print messages)
    problem_type : str
        'coseismic' or 'locking' - determines normalization
    amp : float
        Amplitude for normalization (if problem_type='locking')
        For locking: slip files are in meters, normalize by amp to get coupling ratio
        
    Returns:
    --------
    DataFrame with columns: x, y, z, s_strike, s_dip, slip_mag
    """
    # Read slip values (only 2 columns: s_strike, s_dip)
    slip_data = pd.read_csv(filename, sep=r'\s+', header=None,
                           names=['s_strike', 's_dip'])
    
    # Combine with geometry
    result = fault_geom.copy()
    result['s_strike'] = slip_data['s_strike'].values
    result['s_dip'] = slip_data['s_dip'].values
    
    # Compute slip magnitude
    result['slip_mag'] = np.sqrt(result['s_strike']**2 + result['s_dip']**2)
    
    # Apply normalization for locking problems (ALL files, not just ground truth)
    if problem_type == 'locking':
        if amp is None:
            raise ValueError("amp parameter required for locking problem normalization")
        cols_to_convert = ['s_strike', 's_dip', 'slip_mag']
        result[cols_to_convert] = result[cols_to_convert] / amp
        file_type = "ground truth" if is_ground_truth else "inferred"
        print(f"  Applied locking normalization ({file_type}, divided by amp={amp:.4f})")
    
    print(f"  Loaded: {len(result)} nodes")
    print(f"  Slip range: [{result['slip_mag'].min():.4f}, {result['slip_mag'].max():.4f}]")
    print(f"  s_dip range: [{result['s_dip'].min():.4f}, {result['s_dip'].max():.4f}]")
    
    return result

print(f"Slip loading function defined (default: problem_type='locking')")

In [ ]:
# Load true slip
print("Loading true slip (ground truth)...")
mtrue_file = resultpath + f"mtrue_s_fault_{meshname}{slip_str_gt}.txt"
mtrue_s = read_slip_file(mtrue_file, fault_geom, is_ground_truth=True, 
                         problem_type=problem_type, amp=amp)

In [ ]:
print(mtrue_s.head())

In [ ]:
# Load inferred slip - 3D inversion
print("\nLoading 3D inversion slip...")
m_3d_file = resultpath + f"m_s_fault_{meshname}{slip_str_gt}{het3d_str}{inv_str}{het3d_str}.txt"
m_s_3d = read_slip_file(m_3d_file, fault_geom, is_ground_truth=False, 
                        problem_type=problem_type, amp=amp)

In [ ]:
# Load inferred slip - homogeneous inversion
print("\nLoading homogeneous inversion slip...")
m_hom_file = resultpath + f"m_s_fault_{meshname}{slip_str_gt}{het3d_str}{inv_str}{homo_str}.txt"
m_s_hom = read_slip_file(m_hom_file, fault_geom, is_ground_truth=False,
                         problem_type=problem_type, amp=amp)

# Compute slip difference (3D - hom): positive means hom underestimates
# For locking problems, use s_dip (downdip component) which represents coupling
# Store in m_s_3d since it represents deviation from the "correct" 3D result
if problem_type == 'locking':
    m_s_3d['slip_diff'] = m_s_3d['s_dip'] - m_s_hom['s_dip']
    slip_component = 's_dip (downdip coupling)'
else:
    m_s_3d['slip_diff'] = m_s_3d['slip_mag'] - m_s_hom['slip_mag']
    slip_component = 'slip magnitude'

print(f"\nSlip difference (3D - hom), using {slip_component}:")
print(f"  Mean: {m_s_3d['slip_diff'].mean():.4f}")
print(f"  RMS: {np.sqrt((m_s_3d['slip_diff']**2).mean()):.4f}")
print(f"  Range: [{m_s_3d['slip_diff'].min():.4f}, {m_s_3d['slip_diff'].max():.4f}]")
print(f"  Interpretation: positive = hom underestimates, negative = hom overestimates")

In [ ]:
# Add lon, lat, depth for PyGMT plotting
# Convert from local coordinates (x, y, z in meters) to geographic (lon, lat, depth in km)

# Import utility for coordinate transformation (consistent with mesh generation)
import sys
sys.path.append('/home/staff/chao/SSEinv/Nicoya/codes/')
import utils_plot as utp

# Mesh-specific transformation parameters (nicoya3)
# Define the reference point (center of the projection)
# lon0, lat0 = -85.5+360, 10
lon0, lat0 = -85.5, 10
R = 6371.0  # Earth radius in km

def add_geographic_coords(df):
    """
    Add lon, lat, depth columns to dataframe with x, y, z
    Uses the same coordinate transformation as mesh generation
    """
    # Convert using custom transformation (consistent with mesh)
    df['lon'], df['lat'] = utp.inverse_azi_equidist_proj(df['x'] / 1000.0, df['y'] / 1000.0, lon0, lat0)
    
    df['depth'] = -df['z'].values / 1000.0  # Convert z (meters, negative down) to depth (km, positive down)
    return df

# Add geographic coordinates to all slip dataframes
mtrue_s = add_geographic_coords(mtrue_s)
m_s_3d = add_geographic_coords(m_s_3d)
m_s_hom = add_geographic_coords(m_s_hom)

print(f"\nGeographic coordinates added to slip dataframes (using mesh-consistent transformation):")
print(f"  Reference: lon0={lon0}°, lat0={lat0}°")
print(f"  Lon: [{m_s_3d['lon'].min():.3f}, {m_s_3d['lon'].max():.3f}]°")
print(f"  Lat: [{m_s_3d['lat'].min():.3f}, {m_s_3d['lat'].max():.3f}]°")
print(f"  Depth: [{m_s_3d['depth'].min():.1f}, {m_s_3d['depth'].max():.1f}] km")

# Also add geographic coordinates to fault facet centers
fault_lon, fault_lat = utp.inverse_azi_equidist_proj(
    fault_coords[:, 0] / 1000.0, fault_coords[:, 1] / 1000.0, lon0, lat0)
                                    
fault_depth = -fault_coords[:, 2] / 1000.0  # Convert z to depth in km

print(f"\nGeographic coordinates added to fault facet centers:")
print(f"  Lon: [{fault_lon.min():.3f}, {fault_lon.max():.3f}]°")
print(f"  Lat: [{fault_lat.min():.3f}, {fault_lat.max():.3f}]°")
print(f"  Depth: [{fault_depth.min():.1f}, {fault_depth.max():.1f}] km")

## 5. Load Displacement Data at Stations

In [ ]:
# Load observation file
d_obs_file = resultpath + f"d_obs_{meshname}{slip_str_gt}{het3d_str}.txt"
df_obs = pd.read_csv(d_obs_file, sep=r'\s+', header=None,
                     names=['x', 'y', 'z', 'ux', 'uy', 'uz'])

# Load actual GNSS station locations
obs_disp_name = "CKfig6_data_final.csv"
gnss_stations = pd.read_csv(datadir + obs_disp_name, sep=",", skiprows=1,
                            names=['lon', 'lat', 'vx_Car', 'vy_Car', 'vz_Car',
                                   'vx_std_Car', 'vy_std_Car', 'vz_std_Car'])

df_obs['lon'] = gnss_stations['lon'].values
df_obs['lat'] = gnss_stations['lat'].values
df_obs['depth'] = -df_obs['z'].values / 1000.0
df_obs['u_h'] = np.sqrt(df_obs['ux']**2 + df_obs['uy']**2)

print(f"Loaded displacement at {len(df_obs)} stations")
print(f"Horizontal displacement range: {df_obs['u_h'].min():.4f} - {df_obs['u_h'].max():.4f} m")

## 6. Visualize μ on Fault Surface

**Spatial smoothing for visualization:**
- Applies Gaussian smoothing to reduce noise in plots while preserving data for analysis
- Smoothing is done via triangulation-based interpolation onto a regular grid
- Original data remains unchanged for correlation analysis

In [ ]:
import cmcrameri.cm as cmc
from scipy.ndimage import gaussian_filter
from scipy.interpolate import griddata

# Helper function for spatial smoothing
def smooth_fault_data(coords, values, sigma=2.0, grid_spacing=1000.0):
    """
    Apply spatial smoothing to fault data for visualization
    
    Parameters:
    -----------
    coords : ndarray (N, 3)
        Fault vertex coordinates (x, y, z)
    values : ndarray (N,)
        Values to smooth (e.g., mu, delta_mu)
    sigma : float
        Gaussian kernel standard deviation (in grid cells)
        Larger = more smoothing. Typical: 1-3
    grid_spacing : float
        Regular grid spacing in meters. Default 1000m = 1km
    
    Returns:
    --------
    coords_smooth : ndarray
        Coordinates of smoothed data (for plotting)
    values_smooth : ndarray
        Smoothed values
    """
    # Create regular grid
    x_min, x_max = coords[:, 0].min(), coords[:, 0].max()
    y_min, y_max = coords[:, 1].min(), coords[:, 1].max()
    
    xi = np.arange(x_min, x_max, grid_spacing)
    yi = np.arange(y_min, y_max, grid_spacing)
    xi_grid, yi_grid = np.meshgrid(xi, yi)
    
    # Interpolate to regular grid
    values_grid = griddata(coords[:, :2], values, (xi_grid, yi_grid), method='linear')
    
    # Apply Gaussian smoothing
    values_smooth_grid = gaussian_filter(values_grid, sigma=sigma, mode='nearest')
    
    # Flatten for plotting
    mask = ~np.isnan(values_smooth_grid)
    x_smooth = xi_grid[mask]
    y_smooth = yi_grid[mask]
    values_smooth = values_smooth_grid[mask]
    
    coords_smooth = np.column_stack([x_smooth, y_smooth])
    
    return coords_smooth, values_smooth

# Plot mu on fault surface using matplotlib
def plot_fault_mu_matplotlib(fault_coords, mu_values, title="μ on Fault Surface", cmap='viridis', 
                             symmetric=False, clabel='μ (GPa)', smooth=False, sigma=2.0):
    """
    Plot mu on fault surface
    
    Parameters:
    -----------
    smooth : bool
        If True, apply spatial smoothing for visualization
    sigma : float
        Smoothing strength (larger = more smoothing)
    """
    fig, ax = plt.subplots(figsize=(10, 6))

    if smooth:
        # Apply smoothing for visualization
        coords_smooth, mu_smooth = smooth_fault_data(fault_coords, mu_values, sigma=sigma)
        x_km = coords_smooth[:, 0] / 1e3
        y_km = coords_smooth[:, 1] / 1e3
        plot_values = mu_smooth
        title = title + f" (smoothed, σ={sigma})"
    else:
        x_km = fault_coords[:, 0] / 1e3
        y_km = fault_coords[:, 1] / 1e3
        plot_values = mu_values

    if symmetric:
        vmax = np.nanmax(np.abs(plot_values))
        vmin = -vmax
    else:
        vmin, vmax = None, None

    sc = ax.scatter(x_km, y_km, c=plot_values, s=50,
                    cmap=cmap, vmin=vmin, vmax=vmax, alpha=0.8)

    ax.set_xlabel('X (km)', fontsize=12)
    ax.set_ylabel('Y (km)', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label(clabel, fontsize=12)

    plt.tight_layout()
    return fig, ax

# # Usage: Compare with and without smoothing
# print("Without smoothing:")
# fig1 = plot_fault_mu_matplotlib(fault_coords, mu_fault_3d, "μ (3D)", cmap='gist_rainbow_r', smooth=False)
# # parint("\nWith smoothing (σ=2):")
# # fig2 = plot_fault_mu_matplotlib(fault_coords, mu_fault_3d, "μ (3D)", cmap='gist_rainbow_r', smooth=True, sigma=2.0)
# # print("\nWith smoothing (σ=3):")
# fig3 = plot_fault_mu_matplotlib(fault_coords, mu_anomaly, "μ Anomaly (relative to homogeneous)", 
#                                 cmap='cmc.roma', symmetric=True, clabel='Anomly (%)', smooth=False)

## 7. Slip Comparison (PyGMT)

In [ ]:
# Read the plate interface file
plate_file = "plateinterface/nicoya_slab2_100_10_10.txt"
df_plate = utp.parse_plate_interface_file("/home/staff/chao/SSEinv/Nicoya/" + plate_file)
depths = np.array(df_plate['dep'].unique())
# print("depths:", depths)

# Read the plate file in GMT grd format
grd_file = "/home/staff/chao/SSEinv/Nicoya/plateinterface/cam_slab2_dep_02.24.18.grd"

m2mm = 1e3  # meters to millimeters conversion factor

region=[-87, -84, 8.5, 11.5]    # suitable region for chopping the plate interface grid file 
# region=[-86.75, -84.4, 8.75, 11.25]    # suitable region for chopping the plate interface grid file 
# region=[-88, -83, 6, 14]    # suitable region for chopping the plate interface grid file 

noise_std_h = 0.5 * (gnss_stations['vx_std_Car'].mean() + gnss_stations['vy_std_Car'].mean()) 
noise_std_v = gnss_stations['vz_std_Car'].mean()
error_e, error_n, error_v  = noise_std_h, noise_std_h, noise_std_v

# Read the trench file
trench_file = "plateinterface/trench.gmt.inuse"
# Filter by both longitude and latitude
segments = utp.parse_trench_file("/home/staff/chao/SSEinv/Nicoya/" + trench_file, 
                           lon_range=(region[0], region[1]), 
                           lat_range=(region[2], region[3]))
# Convert to DataFrame for analysis
trench = utp.segments_to_dataframe(segments)

In [ ]:
def plot_slip_comparison_pygmt(m_3d, m_hom, col2plt='s_dip', title_suffix='Coupling', cmap='viridis'):
    """Plot 3-panel slip comparison: 3D inv, Hom inv, Difference"""

    # Define region
    # region = [m_3d['lon'].min()-0.3, m_3d['lon'].max()+0.3,
    #         m_3d['lat'].min()-0.3, m_3d['lat'].max()+0.3]

    fig = pygmt.Figure()

    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False,
                    frame=["a1f0.2", "WSne"], margins=["0.1c", "0.2c"], sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold",
                    MAP_TITLE_OFFSET="-0.2c", FONT_ANNOT_PRIMARY="6p")

        # Common settings
        maxval = max(m_3d[col2plt].max(), m_hom[col2plt].max())
        style = "c0.09c"

        # Panel 1: 3D inversion
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+t3D inv."])
            pygmt.makecpt(cmap=cmap, series=[0, maxval, maxval/20], reverse=True)
            fig.plot(x=m_3d['lon'], y=m_3d['lat'], style=style, fill=m_3d[col2plt], cmap=True)
            fig.coast(shorelines="0.25p,black", area_thresh=20)
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", f"x+l{title_suffix}"])
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.25p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            
        # Panel 2: Homogeneous inversion
        with fig.set_panel(panel=[0, 1]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tHom. inv."])
            pygmt.makecpt(cmap=cmap, series=[0, maxval, maxval/20], reverse=True)
            fig.plot(x=m_hom['lon'], y=m_hom['lat'], style=style, fill=m_hom[col2plt], cmap=True)
            fig.coast(shorelines="0.25p,black", area_thresh=20)
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", f"x+l{title_suffix}"])
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.25p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            
        # Panel 3: Difference
        with fig.set_panel(panel=[0, 2]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+t3D - Hom"])
            diff = m_3d[col2plt] - m_hom[col2plt]
            maxdiff = diff.abs().max()
            maxdiff = 0.6
            pygmt.makecpt(cmap="roma", series=[-maxdiff, maxdiff, maxdiff/10], reverse=False)
            fig.plot(x=m_3d['lon'], y=m_3d['lat'], style=style, fill=diff, cmap=True)
            fig.coast(shorelines="0.25p,black", area_thresh=20)
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lDifference"])
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.25p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")

    return fig

# Usage
if problem_type == 'locking':
    fig_slip_compare = plot_slip_comparison_pygmt(m_s_3d, m_s_hom, col2plt='s_dip',
                                                    title_suffix='Coupling', cmap='viridis')
else:
    fig_slip_compare = plot_slip_comparison_pygmt(m_s_3d, m_s_hom, col2plt='slip_mag',
                                                    title_suffix='Slip (m)', cmap='viridis')

fig_slip_compare.show()
fig_slip_compare.savefig(resultpath + 'slip_comparison_3panel.png', dpi=300)

## 8. Displacement at Stations (PyGMT)

In [ ]:
# Add helper function
def build_disp_vector(u_true, scaleunit, error_e=None, error_n=None, error_v=None):
    # Horizontal components
    df_obs_h_data = {
        "x": u_true['lon'],
        "y": u_true['lat'],
        "east_velocity": u_true['ux']*scaleunit,
        "north_velocity": u_true['uy']*scaleunit,
    }

    # Add error columns only if errors are provided
    if error_e is not None and error_n is not None:
        df_obs_h_data["east_sigma"] = error_e*scaleunit
        df_obs_h_data["north_sigma"] = error_n*scaleunit
        df_obs_h_data["correlation_EN"] = 0.0

    df_obs_h = pd.DataFrame(data=df_obs_h_data)

    # Vertical components
    df_obs_v_data = {
        "x": u_true['lon'],
        "y": u_true['lat'],
        "east_velocity": 0.0,
        "north_velocity": u_true['uz']*scaleunit,
    }

    # Add error columns only if errors are provided
    if error_v is not None:
        df_obs_v_data["east_sigma"] = error_v*scaleunit
        df_obs_v_data["north_sigma"] = error_v*scaleunit
        df_obs_v_data["correlation_EN"] = 0.0

    df_obs_v = pd.DataFrame(data=df_obs_v_data)

    return df_obs_h, df_obs_v

In [ ]:
def plot_slip_and_displacement(mtrue_s, df_obs, trench, df_plate, region, error_e=None, error_n=None, error_v=None, scale_vec=500):
    m2mm = 1000
    col2plt = 's_dip'
    
    df_obs_h, df_obs_v = build_disp_vector(df_obs, m2mm, error_e, error_n, error_v)
    
    fig = pygmt.Figure()
    
    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False,
                     frame=["a1f0.2", "WSne"], margins=["0.1c", "0.2c"], sharey=True):
        
        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black",
                     MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                     FONT_ANNOT_PRIMARY="6p", FONT_ANNOT_SECONDARY="10p")
        
        # Panel 1: True slip
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tTrue slip"])
            maxslip = 1 * amp * m2mm
            pygmt.makecpt(cmap="viridis", series=[0, maxslip, maxslip/10], background=True, reverse=True)
            fig.plot(x=mtrue_s['lon'], y=mtrue_s['lat'], style="c0.08c",
                     fill=mtrue_s[col2plt]*amp*m2mm, cmap=True)
            
            scale_lon, scale_lat = region[1]-0.4, region[-1]-0.2
            mapscale_str = f"g{scale_lon}/{scale_lat}c{scale_lat}+w50k"
            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.25p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            fig.plot(x=df_obs['lon'], y=df_obs['lat'], style="s0.15c",
                    fill="cyan", pen="0.25p,black")
            
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lSlip mag.", "y+l(mm)"])
        
        # Panel 2: Horizontal displacement
        with fig.set_panel(panel=[0, 1]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tSynthetic horiz. disp."])
            fig.coast(shorelines="0.25p,black", area_thresh=20, land="lightgray",
                     water=None, resolution="h", map_scale=mapscale_str)
            
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.25p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            
            fig.velo(data=df_obs_h, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black",
                    spec=f"e{0.5/scale_vec}/0.39",
                    vector="0.1c+a45+p1p,black+ea+gblack+h0")
            
            # Legend
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2",
                    fill="white", pen="0.5p,black", transparency=30)
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c",
                    fill="cyan", pen="0.25p,black")
            
            lgdobs = pd.DataFrame({
                "x": [region[0]+0.15], "y": [region[-2]+0.5],
                "east_velocity": [1], "north_velocity": [0],
                "east_sigma": [1/5], "north_sigma": [1/5], "correlation_EN": [0.0]
            })
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black",
                    spec="e0.5/0.39", vector="0.1c+a45+p1p,black+ea+gblack+h0")
            
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8,
                    font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs H", x=region[0]+0.6, y=region[-2]+0.5,
                    font="8p,Helvetica,black", justify="ML")
            fig.text(text=f"{int(scale_vec)}±{int(scale_vec/5)} mm",
                    x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")
        
        # Panel 3: Vertical displacement
        with fig.set_panel(panel=[0, 2]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tSynthetic vert. disp."])
            fig.coast(shorelines="0.25p,black", area_thresh=20, land="lightgray",
                     water=None, resolution="h", map_scale=mapscale_str)
            
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.25p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            
            fig.velo(data=df_obs_v, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black",
                    spec=f"e{0.5/scale_vec}/0.39",
                    vector="0.1c+a45+p1p,black+ea+gblack+h0")
            
            # Legend
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2",
                    fill="white", pen="0.5p,black", transparency=30)
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c",
                    fill="cyan", pen="0.25p,black")
            
            lgdobs = pd.DataFrame({
                "x": [region[0]+0.4], "y": [region[-2]+0.3],
                "east_velocity": [0], "north_velocity": [1],
                "east_sigma": [1/5], "north_sigma": [1/5], "correlation_EN": [0.0]
            })
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black",
                    spec="e0.5/0.39", vector="0.1c+a45+p1p,black+ea+gblack+h0")
            
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8,
                    font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs V", x=region[0]+0.6, y=region[-2]+0.5,
                    font="8p,Helvetica,black", justify="ML")
            fig.text(text=f"{int(scale_vec)}±{int(scale_vec/5)} mm",
                    x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")
    
    return fig

u_h_max = df_obs['u_h'].max() * 1000
n = np.ceil(u_h_max / 10)
scale_vec = np.round(5*n)
scale_vec = 20
print(u_h_max, scale_vec)

fig = plot_slip_and_displacement(mtrue_s, df_obs, trench, df_plate, region, 
                                  error_e, error_n, error_v, scale_vec)
fig.show()
fig.savefig(resultpath + 'slip_displacement_3panel.png', dpi=300)

## 9. Combined Multi-Panel Figure

In [ ]:
# Combined multi-panel figure (PyGMT) with panels (a)-(d) implemented
lon_stack = np.concatenate([
    np.asarray(fault_lon, dtype=float),
    m_s_3d['lon'].values.astype(float),
    df_obs['lon'].values.astype(float)
])
lat_stack = np.concatenate([
    np.asarray(fault_lat, dtype=float),
    m_s_3d['lat'].values.astype(float),
    df_obs['lat'].values.astype(float)
])
pad_deg = 0.25
region_fault = [
    float(lon_stack.min() - pad_deg),
    float(lon_stack.max() + pad_deg),
    float(lat_stack.min() - pad_deg),
    float(lat_stack.max() + pad_deg)
]

region_fault = region

if problem_type == 'locking':
    slip_diff_label_short = 'Δ Coupling'
    slip_diff_label_long = 'Coupling Difference (3D - Hom)'
    slip_diff_label = 'Δ Coupling, mean removed'
else:
    slip_diff_label_short = 'Δ Slip'
    slip_diff_label_long = 'Slip Difference (3D - Hom)'

slip_diff_values = m_s_3d['slip_diff'].values

scaleunit_mm = 1000.0
# Build horizontal displacement vectors (vertical array used directly below)
df_obs_h_panel, _ = build_disp_vector(df_obs, scaleunit_mm, error_e, error_n, error_v)

# Use scale_vec_panel = 20 as user specified
scale_vec_panel = 20

vertical_mm = df_obs['uz'].values * scaleunit_mm
vert_vmax = max(float(np.nanmax(np.abs(vertical_mm))), 1.0)
lon_center = 0.5 * (region_fault[0] + region_fault[1])
lat_center = 0.5 * (region_fault[2] + region_fault[3])

fig = pygmt.Figure()

# Shared helpers for subplot panels
def _panel_basemap(title):
    fig.basemap(region=region_fault, projection="M?", frame=["a1f0.2", f"+t{title}"])

def _panel_overlays(fill_land=False):
    coast_kwargs = dict(
        shorelines="0.25p,black",
        area_thresh=20,
        land="lightgray" if fill_land else None,
        water=None,
        resolution="h"
    )
    fig.coast(**coast_kwargs)
    fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.25p,darkgray") 
    fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")

def _panel_placeholder(title, message):
    _panel_basemap(title)
    _panel_overlays(fill_land=False)
    fig.text(x=lon_center, y=lat_center, text=message,
             font="9p,Helvetica,gray35", justify="CM")

# Use same style as slip plots
style_fault = "c0.11c"

with fig.subplot(nrows=2, ncols=3, figsize=("21c", "15c"), autolabel=False,
                 sharex=True, sharey=True, margins="0.45c/0.6c"):
    # Fix title offset to prevent overlap with tick labels
    pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold",
                 MAP_TITLE_OFFSET="0.3c", FONT_ANNOT_PRIMARY="7p")

    # Panel (a): μ on the fault from the 3D model
    with fig.set_panel(panel=[0, 0]):
        _panel_basemap("(a) μ on Fault (3D)")
        mu_min = float(np.nanmin(mu_fault_3d))
        mu_max = float(np.nanmax(mu_fault_3d))
        mu_range = max(mu_max - mu_min, 1e-3)
        mu_step = mu_range / 20.0
        print(f"μ range: [{mu_min:.3f}, {mu_max:.3f}] GPa, step: {mu_step:.4f} GPa")

        # Discontinuous colorbar with 20 bins, just like plot_slip_comparison_pygmt
        pygmt.makecpt(cmap="rainbow", series=[mu_min, mu_max, mu_step], background=True, reverse=False)
        fig.plot(x=fault_lon, y=fault_lat, style=style_fault,
                 fill=mu_fault_3d, cmap=True)
        _panel_overlays(fill_land=False)
        # Increased colorbar tick label size from 8p to 9p
        with pygmt.config(FONT_ANNOT_PRIMARY="10p"):
            fig.colorbar(frame=["af", "x+lμ (GPa)"])

    # Panel (b): μ anomaly (% difference relative to homogeneous)
    with fig.set_panel(panel=[0, 1]):
        _panel_basemap("(b) μ Anomaly (%)")

        anom_min = float(np.nanmin(mu_anomaly))
        anom_max = float(np.nanmax(mu_anomaly))
        print(f"Anomaly range: [{anom_min:.3f}, {anom_max:.3f}] %")

        # mu_anomaly_rmean = (mu_fault_3d - np.mean(mu_fault_3d)) / np.mean(mu_fault_3d) * 100
        mu_anomaly_rmean = mu_anomaly - np.mean([anom_min, anom_max])

        mu_anomaly_plt = mu_anomaly
        # mu_anomaly_plt = mu_anomaly_rmean

        anom_vmax = max(float(np.nanmax(np.abs(mu_anomaly_plt))), 1e-2)
        anom_step = anom_vmax / 10.0
        print(f"Anomaly range: ±{anom_vmax:.3f} %, step: {anom_step:.4f} %")

        # Discontinuous colorbar with 20 bins
        pygmt.makecpt(cmap="roma", series=[-anom_vmax, anom_vmax, anom_step], background=True, reverse=False)
        fig.plot(x=fault_lon, y=fault_lat, style=style_fault,
                 fill=mu_anomaly_plt, cmap=True)
        _panel_overlays(fill_land=False)
        # Increased colorbar tick label size from 8p to 9p
        with pygmt.config(FONT_ANNOT_PRIMARY="10p"):
            fig.colorbar(frame=["af", "x+lAnomaly (%)"])

    # Panel (c): slip/coupling bias (3D - homogeneous)
    with fig.set_panel(panel=[0, 2]):
        _panel_basemap(f"(c) {slip_diff_label_long}")

        slip_diff_min = float(np.nanmin(slip_diff_values))
        slip_diff_max = float(np.nanmax(slip_diff_values))
        print(f"Slip diff range: [{slip_diff_min:.3f}, {slip_diff_max:.3f}]")

        slip_diff_values_dmean = slip_diff_values - np.mean([slip_diff_min, slip_diff_max])

        # slip_diff_plt = slip_diff_values
        slip_diff_plt = slip_diff_values_dmean
    
        slip_vmax = max(float(np.nanmax(np.abs(slip_diff_plt))), 1e-4)
        # slip_vmax = 0.52
        if slip_diff_plt is slip_diff_values:
            slip_vmax = 0.6
        elif slip_diff_plt is slip_diff_values_dmean:
            slip_vmax = 0.5
        
        slip_step = slip_vmax / 10.0
        print(f"Anomaly range: ±{slip_vmax:.3f} %, step: {slip_step:.4f}")

        # Discontinuous colorbar with 20 bins
        pygmt.makecpt(cmap="roma", series=[-slip_vmax, slip_vmax, slip_step], background=True, reverse=False)
        fig.plot(x=m_s_3d['lon'], y=m_s_3d['lat'], style=style_fault,
                 fill=slip_diff_plt, cmap=True)
        _panel_overlays(fill_land=False)
        # Increased colorbar tick label size from 8p to 9p
        with pygmt.config(FONT_ANNOT_PRIMARY="10p"):
            if slip_diff_plt is slip_diff_values:
                fig.colorbar(frame=["af", f"x+l{slip_diff_label_short}"])
            elif slip_diff_plt is slip_diff_values_dmean:
                fig.colorbar(frame=["af", f"x+l{slip_diff_label}"])

    # Panel (d): synthetic surface displacement vectors + vertical signal
    with fig.set_panel(panel=[1, 0]):
        _panel_basemap("(d) Surface Displacement")
        vert_step = vert_vmax / 10.0
        # Discontinuous colorbar with 20 bins
        pygmt.makecpt(cmap="vik", series=[-vert_vmax, vert_vmax, vert_step], background=True, reverse=True)
        _panel_overlays(fill_land=True)
        fig.plot(x=df_obs['lon'], y=df_obs['lat'], style="s0.18c",
                 fill=vertical_mm, cmap=True, pen="0.25p,black")
        fig.velo(data=df_obs_h_panel, pen="0.4p,black", line="0.25p,black",
                 spec=f"e{0.5/scale_vec_panel}/0.39",
                 vector="0.1c+a45+p0.75p,black+ea+gblack+h0")
        # Increased colorbar tick label size from 8p to 9p
        with pygmt.config(FONT_ANNOT_PRIMARY="10p"):
            fig.colorbar(frame=["af", "x+lVertical (mm)"])
        
        # Legend - place at bottom left corner, similar to Cell 29
        legend_x = region_fault[0] + 0.15
        legend_y = region_fault[2] + 0.35  # Bottom of panel + small offset
        
        # White box background for legend
        fig.plot(x=legend_x + 0.45, y=legend_y + 0.1, style="r2.3/1.5",
                 fill="white", pen="0.4p,black", transparency=20)
        
        # Station symbol
        fig.plot(x=legend_x + 0.15, y=legend_y + 0.35, style="s0.15c",
                 fill="cyan", pen="0.25p,black")
        
        # Example displacement vector
        legend_vec = pd.DataFrame({
            "x": [legend_x + 0.05],
            "y": [legend_y + 0.15],
            "east_velocity": [scale_vec_panel],
            "north_velocity": [0.0],
            "east_sigma": [scale_vec_panel / 5.0],
            "north_sigma": [scale_vec_panel / 5.0],
            "correlation_EN": [0.0]
        })
        fig.velo(data=legend_vec, pen="0.4p,black", line="0.25p,black",
                 spec=f"e{0.5/scale_vec_panel}/0.39",
                 vector="0.1c+a45+p0.75p,black+ea+gblack+h0")
        
        # Legend text
        fig.text(text="Stations", x=legend_x + 0.35, y=legend_y + 0.35,
                 font="7p,Helvetica,black", justify="ML")
        fig.text(text=f"Obs H", x=legend_x + 0.35, y=legend_y + 0.15,
                 font="7p,Helvetica,black", justify="ML")
        fig.text(text=f"{int(scale_vec_panel)}±{int(scale_vec_panel/5)} mm",
                 x=legend_x + 0.05, y=legend_y - 0.05,
                 font="7p,Helvetica,black", justify="ML")

    # Panel (e): placeholder for Δμ map once contrasts are computed later in the notebook
    with fig.set_panel(panel=[1, 1]):
        _panel_placeholder("(e) μ Contrast Across Fault (TBD)",
                           "Δμ panel will be populated with PyGMT once computed.")

    # Panel (f): placeholder for correlation / stats panel (still Matplotlib-based)
    with fig.set_panel(panel=[1, 2]):
        _panel_placeholder("(f) Correlation / Binned Stats",
                           "Structure-slip correlation plots stay in Matplotlib.")

fig.show()
fig.savefig(resultpath + 'slip_mu_relation_combined.png', dpi=300)
print(f"Figure saved to: {resultpath}slip_mu_relation_combined.png")

## 10. Δμ Across Fault Calculation

Now we implement the calculation of μ contrast across the fault:
- Hanging wall (HW) = overriding plate = blockright
- Footwall (FW) = subducting plate = blockleft
- Δμ = μ_HW - μ_FW

**Layer-based sampling approach:**
Instead of sampling μ at a single point on each side of the fault, we sample multiple points within a thin layer parallel to the fault interface and compute the depth-averaged modulus. This provides:
1. More robust estimates less sensitive to local heterogeneity
2. Better representation of the effective μ structure affecting Green's functions
3. Volume averaging rather than point sampling

**Implementation:**
- Define a thin layer on each side (e.g., 10 km thick)
- Sample at multiple offsets within the layer (e.g., 5 points)
- Average μ values across all samples in each layer
- Compute Δμ from the layer-averaged values

In [ ]:
# Layer-based sampling parameters
# Sample a thicker layer with more points to reduce noise from local heterogeneities
# layer_center_distance = 7500.0     # meters (7.5 km from fault to layer center)
layer_center_distance = 10000.0     # meters (7.5 km from fault to layer center)
layer_thickness = 15000.0          # meters (15 km thick layer)
n_samples_per_layer = 20           # Number of sampling points within each layer (for interpolation)

print(f"Subdomain IDs: blockleft={blockleft} (FW), blockright={blockright} (HW)")
print(f"\nLayer-based sampling configuration:")
print(f"  Layer center: {layer_center_distance/1e3:.1f} km from fault")
print(f"  Layer thickness: {layer_thickness/1e3:.1f} km")
print(f"  Sampling range: [{(layer_center_distance - layer_thickness/2)/1e3:.1f}, "
      f"{(layer_center_distance + layer_thickness/2)/1e3:.1f}] km from fault")

print("\n" + "="*70)
print("COMPUTING BOTH METHODS FOR COMPARISON")
print("="*70)

# # Diagnostic: Check mesh point distribution around fault
# print("\n" + "="*70)
# print("MESH POINT DISTRIBUTION ANALYSIS")
# print("="*70)

# from scipy.spatial import cKDTree

# # Build tree for mesh points
# tree_mesh = cKDTree(mu_3d_grid.points)

# # Sample a few fault points to check
# test_indices = np.linspace(0, len(fault_coords)-1, 20, dtype=int)

# print("\nChecking mesh point density along normals from fault:")
# print(f"{'Fault idx':<10} {'Side':<4} {'Distance (km)':<15} {'Nearest mesh (m)':<20} {'# points <5km'}")
# print("-" * 80)

# for idx in test_indices:
#     coord = fault_coords[idx]
#     normal = fault_normals[idx]

#     # Test at different distances along normal
#     for dist_km in [5, 10, 15, 20, 25]:
#         dist_m = dist_km * 1e3

#         # HW side
#         point_hw = coord + dist_m * normal
#         nearest_dist_hw, _ = tree_mesh.query(point_hw, k=1)
#         n_within_5km_hw = len(tree_mesh.query_ball_point(point_hw, r=5000))

#         # FW side  
#         point_fw = coord - dist_m * normal
#         nearest_dist_fw, _ = tree_mesh.query(point_fw, k=1)
#         n_within_5km_fw = len(tree_mesh.query_ball_point(point_fw, r=5000))

#         print(f"{idx:<10} {'HW':<4} {dist_km:<15} {nearest_dist_hw:<20.1f} {n_within_5km_hw}")
#         print(f"{idx:<10} {'FW':<4} {dist_km:<15} {nearest_dist_fw:<20.1f} {n_within_5km_fw}")

# # Check overall mesh statistics
# print("\n" + "="*70)
# print("MESH STATISTICS")
# print("="*70)

# # Compute approximate mesh spacing by finding nearest neighbor distances
# sample_pts = mu_3d_grid.points[::100]  # Sample every 100th point
# distances_to_neighbors = []
# tree_sample = cKDTree(mu_3d_grid.points)

# for pt in sample_pts[:1000]:  # Check 1000 points
#     dists, _ = tree_sample.query(pt, k=2)  # k=2 to get nearest neighbor (excluding itself)
#     distances_to_neighbors.append(dists[1])

# distances_to_neighbors = np.array(distances_to_neighbors)

# print(f"Mesh element size statistics (from nearest neighbor distances):")
# print(f"  Min:    {distances_to_neighbors.min()/1e3:.2f} km")
# print(f"  Median: {np.median(distances_to_neighbors)/1e3:.2f} km")
# print(f"  Mean:   {distances_to_neighbors.mean()/1e3:.2f} km")
# print(f"  Max:    {distances_to_neighbors.max()/1e3:.2f} km")
# print(f"  95th percentile: {np.percentile(distances_to_neighbors, 95)/1e3:.2f} km")

# print(f"\nTotal mesh points: {len(mu_3d_grid.points)}")
# print(f"Total fault points: {len(fault_coords)}")

# # Check if mesh spacing is larger than search distance at far distances
# if np.median(distances_to_neighbors) > 5000:
#     print(f"\nWARNING: Median mesh spacing ({np.median(distances_to_neighbors)/1e3:.2f} km) > search radius (5 km)")
#     print("This explains why many samples are returning NaN!")
#     print(f"Recommended: Increase max_search_dist to at least {np.percentile(distances_to_neighbors, 95)/1e3:.1f} km")


In [ ]:
# # METHOD 1: Linear interpolation (smooth, artificial)
# print("\n### METHOD 1: Linear Interpolation ###")
# mu_hw_interp, mu_fw_interp = sample_mu_across_fault_layered(
#     mu_3d_grid, fault_coords, fault_normals,
#     layer_center_distance, layer_thickness, n_samples_per_layer,
#     method='linear'
# )
# delta_mu_interp = mu_hw_interp - mu_fw_interp

# METHOD 2: Actual mesh vertices (true FEniCS values)
print("\n### METHOD 2: Actual Mesh Vertices ###")
mu_hw_mesh, mu_fw_mesh = sample_mu_across_fault_layered(
    mu_3d_grid, fault_coords, fault_normals,
    layer_center_distance, layer_thickness, n_samples_per_layer,
    field_name='shear modulus', 
    method='mesh_vertices'
)

delta_mu_mesh = mu_hw_mesh - mu_fw_mesh

print("\n" + "="*70)
print("COMPARISON OF METHODS")
print("="*70)

# print(f"\nΔμ (Linear Interpolation):")
# print(f"  Mean: {np.nanmean(delta_mu_interp):.2f} GPa")
# print(f"  Std:  {np.nanstd(delta_mu_interp):.2f} GPa")
# print(f"  Range: [{np.nanmin(delta_mu_interp):.2f}, {np.nanmax(delta_mu_interp):.2f}] GPa")

print(f"\nΔμ (Mesh Vertices - True FEniCS):")
print(f"  Mean: {np.nanmean(delta_mu_mesh):.2f} GPa")
print(f"  Std:  {np.nanstd(delta_mu_mesh):.2f} GPa")
print(f"  Range: [{np.nanmin(delta_mu_mesh):.2f}, {np.nanmax(delta_mu_mesh):.2f}] GPa")

# print(f"\nDifference between methods:")
# diff_methods = delta_mu_interp - delta_mu_mesh
# print(f"  Mean difference: {np.nanmean(diff_methods):.2f} GPa")
# print(f"  RMS difference:  {np.sqrt(np.nanmean(diff_methods**2)):.2f} GPa")


In [ ]:
### set the default 

# Default to mesh vertices method (true FEniCS values) for main analysis
mu_hw = mu_hw_mesh.copy()
mu_fw = mu_fw_mesh.copy()
delta_mu = delta_mu_mesh.copy()

# print(f"\n>>> Using MESH VERTICES method as default (delta_mu)")
# print(f">>> Interpolation results saved as delta_mu_interp for comparison")

# # Default to interpolated method for smoother results
# mu_hw = mu_hw_interp.copy()
# mu_fw = mu_fw_interp.copy()
# delta_mu = delta_mu_interp.copy()

In [ ]:
# # Comparison plot: Mesh vertices vs Linear interpolation (with optional smoothing)
# def plot_method_comparison(fault_coords, delta_mu_mesh, delta_mu_interp, smooth=False, sigma=2.0):
#     """Compare mesh vertices and linear interpolation methods"""
#     fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
#     if smooth:
#         # Apply smoothing for visualization
#         coords_smooth_mesh, delta_mu_mesh_plot = smooth_fault_data(fault_coords, delta_mu_mesh, sigma=sigma)
#         coords_smooth_interp, delta_mu_interp_plot = smooth_fault_data(fault_coords, delta_mu_interp, sigma=sigma)
#         x_km_mesh = coords_smooth_mesh[:, 0] / 1e3
#         y_km_mesh = coords_smooth_mesh[:, 1] / 1e3
#         x_km_interp = coords_smooth_interp[:, 0] / 1e3
#         y_km_interp = coords_smooth_interp[:, 1] / 1e3
#         title_suffix = f' (smoothed, σ={sigma})'
#     else:
#         x_km_mesh = fault_coords[:, 0] / 1e3
#         y_km_mesh = fault_coords[:, 1] / 1e3
#         x_km_interp = x_km_mesh
#         y_km_interp = y_km_mesh
#         delta_mu_mesh_plot = delta_mu_mesh
#         delta_mu_interp_plot = delta_mu_interp
#         title_suffix = ''
    
#     # Find common color scale
#     vmax = max(np.nanmax(np.abs(delta_mu_mesh_plot)), np.nanmax(np.abs(delta_mu_interp_plot)))
    
#     # Panel 1: Mesh vertices (true FEniCS)
#     sc1 = axes[0].scatter(x_km_mesh, y_km_mesh, c=delta_mu_mesh_plot, s=50,
#                           cmap='RdBu_r', vmin=-vmax, vmax=vmax, alpha=0.8)
#     axes[0].set_xlabel('X (km)')
#     axes[0].set_ylabel('Y (km)')
#     axes[0].set_title('Δμ (Mesh Vertices [True FEniCS])' + title_suffix, fontsize=12, fontweight='bold')
#     axes[0].set_aspect('equal')
#     axes[0].grid(True, alpha=0.3)
#     cbar1 = plt.colorbar(sc1, ax=axes[0])
#     cbar1.set_label('Δμ (GPa)')
#     axes[0].text(0.02, 0.98, f'Mean = {np.nanmean(delta_mu_mesh):.2f} GPa\nStd = {np.nanstd(delta_mu_mesh):.2f} GPa',
#                  transform=axes[0].transAxes, fontsize=9, va='top',
#                  bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
#     # Panel 2: Linear interpolation
#     sc2 = axes[1].scatter(x_km_interp, y_km_interp, c=delta_mu_interp_plot, s=50,
#                           cmap='RdBu_r', vmin=-vmax, vmax=vmax, alpha=0.8)
#     axes[1].set_xlabel('X (km)')
#     axes[1].set_ylabel('Y (km)')
#     axes[1].set_title('Δμ (Linear Interpolation [Smoothed])' + title_suffix, fontsize=12, fontweight='bold')
#     axes[1].set_aspect('equal')
#     axes[1].grid(True, alpha=0.3)
#     cbar2 = plt.colorbar(sc2, ax=axes[1])
#     cbar2.set_label('Δμ (GPa)')
#     axes[1].text(0.02, 0.98, f'Mean = {np.nanmean(delta_mu_interp):.2f} GPa\nStd = {np.nanstd(delta_mu_interp):.2f} GPa',
#                  transform=axes[1].transAxes, fontsize=9, va='top',
#                  bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
#     # Panel 3: Difference
#     diff = delta_mu_mesh - delta_mu_interp
#     if smooth:
#         coords_smooth_diff, diff_plot = smooth_fault_data(fault_coords, diff, sigma=sigma)
#         x_km_diff = coords_smooth_diff[:, 0] / 1e3
#         y_km_diff = coords_smooth_diff[:, 1] / 1e3
#     else:
#         x_km_diff = x_km_mesh
#         y_km_diff = y_km_mesh
#         diff_plot = diff
    
#     vmax_diff = np.nanmax(np.abs(diff_plot))
#     sc3 = axes[2].scatter(x_km_diff, y_km_diff, c=diff_plot, s=50,
#                           cmap='RdBu_r', vmin=-vmax_diff, vmax=vmax_diff, alpha=0.8)
#     axes[2].set_xlabel('X (km)')
#     axes[2].set_ylabel('Y (km)')
#     axes[2].set_title('Difference (Mesh - Interp)' + title_suffix, fontsize=12, fontweight='bold')
#     axes[2].set_aspect('equal')
#     axes[2].grid(True, alpha=0.3)
#     cbar3 = plt.colorbar(sc3, ax=axes[2])
#     cbar3.set_label('Δμ difference (GPa)')
#     axes[2].text(0.02, 0.98, f'RMS diff = {np.sqrt(np.nanmean(diff**2)):.2f} GPa',
#                  transform=axes[2].transAxes, fontsize=9, va='top',
#                  bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
#     plt.tight_layout()
#     return fig

# # Plot without smoothing
# print("Method comparison WITHOUT smoothing:")
# fig1 = plot_method_comparison(fault_coords, delta_mu_mesh, delta_mu_interp, smooth=False)
# plt.savefig(resultpath + 'delta_mu_method_comparison_raw.png', dpi=300, bbox_inches='tight')
# plt.show()

# # Plot with smoothing
# print("\nMethod comparison WITH smoothing (σ=2):")
# fig2 = plot_method_comparison(fault_coords, delta_mu_mesh, delta_mu_interp, smooth=True, sigma=2.0)
# plt.savefig(resultpath + 'delta_mu_method_comparison_smooth.png', dpi=300, bbox_inches='tight')
# plt.show()

# print(f"\nMethod comparison figures saved to: {resultpath}delta_mu_method_comparison_*.png")
# print(f"\nKey observation:")
# print(f"  - Mesh vertices: Shows true discretization structure from FEniCS")
# print(f"  - Linear interp: Artificially smoothed during sampling")
# print(f"  - Visualization smoothing: Applied ONLY for plotting, data unchanged")
# print(f"  - RMS difference: {np.sqrt(np.nanmean((delta_mu_mesh - delta_mu_interp)**2)):.2f} GPa")

In [ ]:
# # Plot Δμ on fault surface with optional smoothing
# def plot_delta_mu(fault_coords, delta_mu, title='Δμ Across Fault (μ_HW - μ_FW)', 
#                   smooth=False, sigma=2.0, save_suffix=''):
#     """Plot delta mu with optional smoothing"""
#     fig, ax = plt.subplots(figsize=(10, 6))
    
#     if smooth:
#         coords_smooth, delta_mu_smooth = smooth_fault_data(fault_coords, delta_mu, sigma=sigma)
#         x_km = coords_smooth[:, 0] / 1e3
#         y_km = coords_smooth[:, 1] / 1e3
#         plot_values = delta_mu_smooth
#         title = title + f' (smoothed, σ={sigma})'
#     else:
#         x_km = fault_coords[:, 0] / 1e3
#         y_km = fault_coords[:, 1] / 1e3
#         plot_values = delta_mu
    
#     # Plot with diverging colormap
#     vmax = np.nanmax(np.abs(plot_values))
#     sc = ax.scatter(x_km, y_km, c=plot_values, s=50,
#                     cmap='RdBu_r', vmin=-vmax, vmax=vmax, alpha=0.8)
    
#     ax.set_xlabel('X (km)', fontsize=12)
#     ax.set_ylabel('Y (km)', fontsize=12)
#     ax.set_title(title, fontsize=14, fontweight='bold')
#     ax.set_aspect('equal')
#     ax.grid(True, alpha=0.3)
    
#     cbar = plt.colorbar(sc, ax=ax)
#     cbar.set_label('Δμ (GPa)', fontsize=12)
    
#     # Add text annotations
#     ax.text(0.02, 0.98, f'Mean Δμ = {np.nanmean(delta_mu):.2f} GPa',
#             transform=ax.transAxes, fontsize=10, va='top',
#             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
#     plt.tight_layout()
#     if save_suffix:
#         plt.savefig(resultpath + f'delta_mu_fault{save_suffix}.png', dpi=300, bbox_inches='tight')
#     return fig

# # Plot without smoothing
# print("Plotting Δμ without smoothing:")
# fig1 = plot_delta_mu(fault_coords, delta_mu, smooth=False, save_suffix='_raw')
# plt.show()

# # Plot with moderate smoothing
# print("\nPlotting Δμ with smoothing (σ=2):")
# fig2 = plot_delta_mu(fault_coords, delta_mu, smooth=True, sigma=2.0, save_suffix='_smooth_s2')
# plt.show()

# # Plot with stronger smoothing
# print("\nPlotting Δμ with smoothing (σ=3):")
# fig3 = plot_delta_mu(fault_coords, delta_mu, smooth=True, sigma=3.0, save_suffix='_smooth_s3')
# plt.show()

# print(f"\nFigures saved to: {resultpath}delta_mu_fault_*.png")

In [ ]:
# 8-panel PyGMT figure: μ_FW, μ_fault_3d, μ_HW and their anomalies.
lon_stack = np.concatenate([
    np.asarray(fault_lon, dtype=float),
    m_s_3d['lon'].values.astype(float),
    df_obs['lon'].values.astype(float)
])
lat_stack = np.concatenate([
    np.asarray(fault_lat, dtype=float),
    m_s_3d['lat'].values.astype(float),
    df_obs['lat'].values.astype(float)
])
pad_deg = 0.25
region_fault = [
    float(lon_stack.min() - pad_deg),
    float(lon_stack.max() + pad_deg),
    float(lat_stack.min() - pad_deg),
    float(lat_stack.max() + pad_deg)
]
region_fault = region

style_fault = "c0.11c"
colorbar_params = "+o0/-0.75c+w5.8c/0.32c+h"

mu_all = np.concatenate([mu_fw, mu_fault_3d, mu_hw]).astype(float)
mu_min = float(np.nanmin(mu_all))
mu_max = float(np.nanmax(mu_all))
mu_step = max((mu_max - mu_min) / 20.0, 1e-3)
print(f"μ range: [{mu_min:.3f}, {mu_max:.3f}] GPa, step: {mu_step:.4f} GPa")

mu_fw_anom = (mu_fw - mu_ref) / mu_ref * 100.0
mu_fault_anom = (mu_fault_3d - mu_ref) / mu_ref * 100.0
mu_hw_anom = (mu_hw - mu_ref) / mu_ref * 100.0
anom_all = np.concatenate([mu_fw_anom, mu_fault_anom, mu_hw_anom]).astype(float)
anom_vmax = max(float(np.nanmax(np.abs(anom_all))), 1e-2)
anom_step = max(anom_vmax / 10.0, 1e-3)
print(f"Anomaly range: ±{anom_vmax:.3f} %, step: {anom_step:.4f} %")

slip_diff_values = m_s_3d['slip_diff'].values
slip_vmax = max(float(np.nanmax(np.abs(slip_diff_values))), 1e-4)
slip_step = max(slip_vmax / 10.0, 1e-5)
print(f"Slip/Coupling diff range: ±{slip_vmax:.5f}, step: {slip_step:.6f}")

delta_mu_rel_hom = (mu_hw - mu_fw) / mu_ref * 100.0
delta_rel_vmax = max(float(np.nanmax(np.abs(delta_mu_rel_hom))), 1e-2)
delta_rel_step = max(delta_rel_vmax / 10.0, 1e-3)
print(f"Δμ relative to hom range: ±{delta_rel_vmax:.3f} %, step: {delta_rel_step:.4f} %")   

fig = pygmt.Figure()

def _panel_basemap(title):
    fig.basemap(region=region_fault, projection="M?", frame=["a1f0.2", f"+t{title}"])

def _panel_overlays(fill_land=False):
    coast_kwargs = dict(
        shorelines="0.25p,black",
        area_thresh=20,
        land="lightgray" if fill_land else None,
        water=None,
        resolution="h"
    )
    fig.coast(**coast_kwargs)
    fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.25p,darkgray") 
    fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")


with fig.subplot(nrows=2, ncols=4, figsize=("27c", "15c"), autolabel=False,
                 sharex=True, sharey=True, margins="0.45c/0.6c"):
    pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold",
                 MAP_TITLE_OFFSET="0.3c", FONT_ANNOT_PRIMARY="7p")

    mu_titles = ["(a) μ_FW (Layer Avg)", "(b) μ on Fault (3D)",
                 "(c) μ_HW (Layer Avg)", "(d) Δ Coupling (3D - Hom)"]
    mu_fields = [mu_fw, mu_fault_3d, mu_hw, slip_diff_values]
    for idx, (title, data) in enumerate(zip(mu_titles, mu_fields)):
        with fig.set_panel(panel=[0, idx]):
            _panel_basemap(title)
            if idx < 3:
                pygmt.makecpt(cmap="rainbow", series=[mu_min, mu_max, mu_step], background=True)
                fig.plot(x=fault_lon, y=fault_lat, style=style_fault, fill=data, cmap=True)
                label = "μ (GPa)"
            else:
                pygmt.makecpt(cmap="roma", series=[-slip_vmax, slip_vmax, slip_step], background=True)
                fig.plot(x=m_s_3d['lon'], y=m_s_3d['lat'], style=style_fault, fill=data, cmap=True)
                label = "Δ Coupling"
            _panel_overlays(fill_land=False)
            with pygmt.config(FONT_ANNOT_PRIMARY="10p"):
                fig.colorbar(frame=["af", f"x+l{label}"])

    anom_titles = ["(e) μ_FW Anomaly (%)", "(f) μ Fault Anomaly (%)",
                   "(g) μ_HW Anomaly (%)", "(h) Δμ Relative to Hom (%)"]
    anom_fields = [mu_fw_anom, mu_fault_anom, mu_hw_anom, delta_mu_rel_hom]
    for idx, (title, data) in enumerate(zip(anom_titles, anom_fields)):
        with fig.set_panel(panel=[1, idx]):
            _panel_basemap(title)
            if idx < 3:
                pygmt.makecpt(cmap="roma", series=[-anom_vmax, anom_vmax, anom_step], background=True)
                fig.plot(x=fault_lon, y=fault_lat, style=style_fault, fill=data, cmap=True)
                label = "Anomaly (%)"
            else:
                pygmt.makecpt(cmap="roma", series=[-delta_rel_vmax, delta_rel_vmax, delta_rel_step], background=True)
                fig.plot(x=fault_lon, y=fault_lat, style=style_fault, fill=data, cmap=True)
                label = "Δμ (%)"
            _panel_overlays(fill_land=False)
            with pygmt.config(FONT_ANNOT_PRIMARY="10p"):
                fig.colorbar(frame=["af", f"x+l{label}"])   #position=f"JCB{colorbar_params}", 

fig.show()
fig.savefig(resultpath + 'mu_layer_vs_fault_8panel.png', dpi=300)
print(f"Layer-resolved μ figure saved to: {resultpath}mu_layer_vs_fault_8panel.png")


In [ ]:
# 4-panel PyGMT figure: μ_fault_3d, μ_HW-μ_FW difference and their anomalies/deltas
lon_stack = np.concatenate([
    np.asarray(fault_lon, dtype=float),
    m_s_3d['lon'].values.astype(float),
    df_obs['lon'].values.astype(float)
])
lat_stack = np.concatenate([
    np.asarray(fault_lat, dtype=float),
    m_s_3d['lat'].values.astype(float),
    df_obs['lat'].values.astype(float)
])
pad_deg = 0.25
region_fault = [
    float(lon_stack.min() - pad_deg),
    float(lon_stack.max() + pad_deg),
    float(lat_stack.min() - pad_deg),
    float(lat_stack.max() + pad_deg)
]
region_fault = region

style_fault = "c0.1c"
colorbar_params = "+o0/-0.75c+w5.8c/0.32c+h"

mu_all = np.concatenate([mu_fw, mu_fault_3d, mu_hw]).astype(float)
mu_min = float(np.nanmin(mu_all))
mu_max = float(np.nanmax(mu_all))
mu_step = max((mu_max - mu_min) / 20.0, 1e-3)
print(f"μ range: [{mu_min:.3f}, {mu_max:.3f}] GPa, step: {mu_step:.4f} GPa")

mu_fw_anom = (mu_fw - mu_ref) / mu_ref * 100.0
mu_fault_anom = (mu_fault_3d - mu_ref) / mu_ref * 100.0
mu_hw_anom = (mu_hw - mu_ref) / mu_ref * 100.0
anom_all = np.concatenate([mu_fw_anom, mu_fault_anom, mu_hw_anom]).astype(float)
anom_vmax = max(float(np.nanmax(np.abs(anom_all))), 1e-2)
anom_step = max(anom_vmax / 10.0, 1e-3)
print(f"Anomaly range: ±{anom_vmax:.3f} %, step: {anom_step:.4f} %")

slip_diff_values = m_s_3d['slip_diff'].values
slip_vmax = max(float(np.nanmax(np.abs(slip_diff_values))), 1e-4)
slip_step = max(slip_vmax / 10.0, 1e-5)
print(f"Slip/Coupling diff range: ±{slip_vmax:.5f}, step: {slip_step:.6f}")

# Calculate absolute difference μ_HW - μ_FW
mu_hw_minus_fw = mu_hw - mu_fw
mu_diff_vmax = max(float(np.nanmax(np.abs(mu_hw_minus_fw))), 1e-3)
mu_diff_step = max(mu_diff_vmax / 10.0, 1e-4)
print(f"μ_HW - μ_FW range: ±{mu_diff_vmax:.3f} GPa, step: {mu_diff_step:.4f} GPa")

delta_mu_rel_hom = (mu_hw - mu_fw) / mu_ref * 100.0
delta_rel_vmax = max(float(np.nanmax(np.abs(delta_mu_rel_hom))), 1e-2)
delta_rel_step = max(delta_rel_vmax / 10.0, 1e-3)
print(f"Δμ relative to hom range: ±{delta_rel_vmax:.3f} %, step: {delta_rel_step:.4f} %")


fig = pygmt.Figure()

def _panel_basemap(title):
    fig.basemap(region=region_fault, projection="M?", frame=["a1f0.2", f"+t{title}"])

def _panel_overlays(fill_land=False):
    coast_kwargs = dict(
        shorelines="0.25p,black",
        area_thresh=20,
        land="lightgray" if fill_land else None,
        water=None,
        resolution="h"
    )
    fig.coast(**coast_kwargs)
    fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.25p,darkgray") 
    fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")

with fig.subplot(nrows=2, ncols=2, figsize=("12c", "14c"), autolabel=False,
                sharex=True, sharey=True, margins="0.45c/0.6c"):
    pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold",
                MAP_TITLE_OFFSET="0.3c", FONT_ANNOT_PRIMARY="7p")

    # Row 1: Panel (b) and Panel with μ_HW - μ_FW
    # Panel (b): μ on Fault (3D)
    with fig.set_panel(panel=[0, 0]):
        _panel_basemap("(a) μ on Fault (3D)")
        pygmt.makecpt(cmap="rainbow", series=[mu_min, mu_max, mu_step], background=True)
        fig.plot(x=fault_lon, y=fault_lat, style=style_fault, fill=mu_fault_3d,
cmap=True)
        _panel_overlays(fill_land=False)
        with pygmt.config(FONT_ANNOT_PRIMARY="10p"):
            fig.colorbar(frame=["af", "x+lμ (GPa)"])

    # NEW Panel: μ_HW - μ_FW (absolute difference)
    with fig.set_panel(panel=[0, 1]):
        _panel_basemap("(b) μ_HW - μ_FW")
        pygmt.makecpt(cmap="roma", series=[-mu_diff_vmax, mu_diff_vmax, mu_diff_step],
background=True)
        fig.plot(x=fault_lon, y=fault_lat, style=style_fault, fill=mu_hw_minus_fw,
cmap=True)
        _panel_overlays(fill_land=False)
        with pygmt.config(FONT_ANNOT_PRIMARY="10p"):
            fig.colorbar(frame=["af", "x+lΔμ (GPa)"])

    # Row 2: Panel (f) and Panel (h)
    # Panel (f): μ Fault Anomaly (%)
    with fig.set_panel(panel=[1, 0]):
        _panel_basemap("(c) μ Fault Anomaly (%)")
        pygmt.makecpt(cmap="roma", series=[-anom_vmax, anom_vmax, anom_step],
background=True)
        fig.plot(x=fault_lon, y=fault_lat, style=style_fault, fill=mu_fault_anom,
cmap=True)
        _panel_overlays(fill_land=False)
        with pygmt.config(FONT_ANNOT_PRIMARY="10p"):
            fig.colorbar(frame=["af", "x+lAnomaly (%)"])

    # Panel (h): Δμ Relative to Hom (%)
    with fig.set_panel(panel=[1, 1]):
        _panel_basemap("(d) Δμ Relative to Hom (%)")
        pygmt.makecpt(cmap="roma", series=[-delta_rel_vmax, delta_rel_vmax,
delta_rel_step], background=True)
        fig.plot(x=fault_lon, y=fault_lat, style=style_fault, fill=delta_mu_rel_hom,
cmap=True)
        _panel_overlays(fill_land=False)
        with pygmt.config(FONT_ANNOT_PRIMARY="10p"):
            fig.colorbar(frame=["af", "x+lΔμ (%)"])

fig.show()
fig.savefig(resultpath + 'mu_fault_hw_4panel.png', dpi=300)
print(f"4-panel figure saved to: {resultpath}mu_fault_hw_4panel.png")

## 10.5. Horizontal Slices of μ Structure

Visualize how μ and μ anomaly vary with depth using horizontal slices from 0 to 80 km depth.

**Purpose:**
- Understand depth variation of shear modulus structure
- Investigate how μ contrasts change with depth
- Relate depth-dependent structure to slip inversion bias

**Implementation:**
Two approaches for creating horizontal slices:
1. **Direct griddata interpolation**: Fast but can be grainy due to mesh discretization
2. **PyVista slicing** (recommended): Uses PyVista's native slicing for smoother results

**Visualization:**
- 3×3 panel layout showing 9 depth levels (0, -10, -20, ..., -80 km)
- PyGMT style with coastlines, plate interface contours, and trench
- Consistent colormaps: rainbow for μ, roma for anomaly

In [ ]:
def _plot_slice_panels(data_slices, depths_km, cmap, series, label, filename):
    """Plot 3x3 panel of horizontal slices in PyGMT style"""

    fig = pygmt.Figure()
    with fig.subplot(nrows=3, ncols=3, figsize=("21c", "24c"), autolabel=False,
                    sharex=True, sharey=True, margins="0.35c/0.5c", frame=["WSne"]):
        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold",
                    MAP_TITLE_OFFSET="0.3c", FONT_ANNOT_PRIMARY="7p")

        for idx_panel, (depth_km, data_grid) in enumerate(zip(depths_km, data_slices)):
            row = idx_panel // 3
            col = idx_panel % 3

            with fig.set_panel(panel=[row, col]):
                fig.basemap(region=region_fault, projection="M?",
                            frame=["a1f0.2", f"+tDepth = {abs(depth_km):.0f} km"])

                # Create grid from data
                grid = pygmt.xyz2grd(x=lon_grid.ravel(), y=lat_grid.ravel(),
                                    z=data_grid.ravel(), region=region_fault,
                                    spacing=(lon_spacing, lat_spacing))

                # Make colormap and plot
                pygmt.makecpt(cmap=cmap, series=series, background=True)
                fig.grdimage(grid=grid, cmap=True)

                fig.coast(shorelines="0.5p,black", area_thresh=100)

                fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.25p,darkgray")
                # fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")

                # Colorbar (only on right column)
                if col == 2:
                    with pygmt.config(FONT_ANNOT_PRIMARY="9p"):
                        fig.colorbar(frame=["af", f"x+l{label}"])

                # Plot vertical slice locations on first panel
                # Vertical slices are defined in ROTATED coordinates (45° CCW from N-E)
                # where rotx = along-dip (NE), roty = along-strike (NW-SE)
                if idx_panel == 0:
                    # Plot along-strike slice positions
                    # In rotated coords: constant rotx (along-dip), varying roty (along-strike)
                    for rotx_km in x_strike_km:
                        # Create line in rotated coordinates
                        roty_line_km = np.linspace(y_min, y_max, 100)
                        rotx_line_km = np.full_like(roty_line_km, rotx_km)

                        # Un-rotate: rotated coords -> mesh coords (rotate by -45°)
                        x_mesh_km, y_mesh_km = utp.rot_xy(rotx_line_km, roty_line_km, rot=-rot)

                        # Convert from mesh (km) to lon/lat
                        lon_line, lat_line = utp.inverse_azi_equidist_proj(
                            x_mesh_km, y_mesh_km, lon0, lat0
                        )

                        # Plot the line
                        fig.plot(x=lon_line, y=lat_line, pen="0.8p,white,--")

                    # Plot along-dip slice positions
                    # In rotated coords: constant roty (along-strike), varying rotx (along-dip)
                    for roty_km in y_dip_km:
                        # Create line in rotated coordinates
                        rotx_line_km = np.linspace(x_min, x_max, 100)
                        roty_line_km = np.full_like(rotx_line_km, roty_km)

                        # Un-rotate: rotated coords -> mesh coords (rotate by -45°)
                        x_mesh_km, y_mesh_km = utp.rot_xy(rotx_line_km, roty_line_km, rot=-rot)

                        # Convert from mesh (km) to lon/lat
                        lon_line, lat_line = utp.inverse_azi_equidist_proj(
                            x_mesh_km, y_mesh_km, lon0, lat0
                        )

                        # Plot the line
                        fig.plot(x=lon_line, y=lat_line, pen="0.8p,white,-.")

    fig.show()
    
    fig.savefig(filename, dpi=300)
    print(f"Saved: {filename}")
    return fig


In [ ]:
# # Horizontal slices of μ and μ anomaly (3x3 depth panels).
# depth_levels = np.arange(0.0, -80.1, -10.0)
# lon_spacing = 0.01
# lat_spacing = 0.01
# lon_vals = np.arange(region_fault[0], region_fault[1] + lon_spacing, lon_spacing)
# lat_vals = np.arange(region_fault[2], region_fault[3] + lat_spacing, lat_spacing)
# lon_grid, lat_grid = np.meshgrid(lon_vals, lat_vals)
# grid_shape = (len(lat_vals), len(lon_vals))

# from scipy.interpolate import griddata

# x_local, y_local = utp.azi_equidist_proj(lon_grid.ravel(), lat_grid.ravel(), lon0, lat0)

# mu_points = mu_3d_grid.points
# mu_values_all = mu_3d_grid.point_data['shear modulus']
# mu_hom_points = mu_hom_grid.points
# mu_hom_values_all = mu_hom_grid.point_data['shear modulus']

# mu_slice_list = []
# anom_slice_list = []

# for depth_km in depth_levels:
#     z_value = depth_km * 1e3
#     sample_points = np.column_stack([
#         x_local,
#         y_local,
#         np.full_like(x_local, z_value)
#     ])

#     mu_values = griddata(mu_points, mu_values_all, sample_points, method='linear')
#     mu_hom_values = griddata(mu_hom_points, mu_hom_values_all, sample_points, method='linear')

#     mu_grid = mu_values.reshape(grid_shape)
#     mu_hom_grid_slice = mu_hom_values.reshape(grid_shape)
#     mu_anom_grid = (mu_grid - mu_hom_grid_slice) / mu_hom_grid_slice * 100.0

#     mu_slice_list.append(mu_grid)
#     anom_slice_list.append(mu_anom_grid)

# mu_slice_min = float(np.nanmin([np.nanmin(slc) for slc in mu_slice_list]))
# mu_slice_max = float(np.nanmax([np.nanmax(slc) for slc in mu_slice_list]))
# mu_slice_step = max((mu_slice_max - mu_slice_min) / 20.0, 1e-3)

# anom_slice_vmax = float(np.nanmax([np.nanmax(np.abs(slc)) for slc in anom_slice_list]))
# anom_slice_step = max(anom_slice_vmax / 10.0, 1e-3)

# def _plot_slice_panels(data_slices, depths_km, cmap, series, label, filename):
#     """Plot 3x3 panel of horizontal slices in PyGMT style"""
#     fig = pygmt.Figure()
#     with fig.subplot(nrows=3, ncols=3, figsize=("21c", "24c"), autolabel=False,
#                      sharex=True, sharey=True, margins="0.35c/0.5c"):
#         pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold",
#                      MAP_TITLE_OFFSET="0.3c", FONT_ANNOT_PRIMARY="7p")
        
#         for idx_panel, (depth_km, data_grid) in enumerate(zip(depths_km, data_slices)):
#             row = idx_panel // 3
#             col = idx_panel % 3
            
#             with fig.set_panel(panel=[row, col]):
#                 fig.basemap(region=region_fault, projection="M?",
#                             frame=["a1f0.2", f"+tDepth = {abs(depth_km):.0f} km"])
                
#                 # Create grid from data
#                 grid = pygmt.xyz2grd(x=lon_grid.ravel(), y=lat_grid.ravel(),
#                                      z=data_grid.ravel(), region=region_fault,
#                                      spacing=(lon_spacing, lat_spacing))
                
#                 # Make colormap and plot
#                 pygmt.makecpt(cmap=cmap, series=series, background=True)
#                 fig.grdimage(grid=grid, cmap=True)
                
#                 # Add geographic features
#                 fig.coast(shorelines="0.25p,black", area_thresh=20)
#                 fig.grdcontour(grid=plate_grd, levels=5, limit="-100/-5",
#                                annotation="20+f6p", pen="0.4p,darkgray")
#                 fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray")
                
#                 # Colorbar (only on right column)
#                 if col == 2:
#                     with pygmt.config(FONT_ANNOT_PRIMARY="9p"):
#                         fig.colorbar(frame=["af", f"x+l{label}"])
    
#     fig.show()
#     fig.savefig(filename, dpi=300)
#     print(f"Saved slice figure to: {filename}")
    
#     # return fig

# mu_slice_file = resultpath + 'mu_horizontal_slices.png'
# _plot_slice_panels(mu_slice_list, depth_levels, cmap="rainbow",
#                    series=[mu_slice_min, mu_slice_max, mu_slice_step],
#                    label="μ (GPa)", filename=mu_slice_file)

# anom_slice_file = resultpath + 'mu_anomaly_horizontal_slices.png'
# _plot_slice_panels(anom_slice_list, depth_levels, cmap="roma",
#                    series=[-anom_slice_vmax, anom_slice_vmax, anom_slice_step],
#                    label="μ anomaly (%)", filename=anom_slice_file)



In [ ]:
# # Mesh 1 parameters
# lon0_CK, lat0_CK = -84, 7
# rot_CK = 45.0
# x0_CK, y0_CK = 130e3, 350e3  # meters

# # # Mesh 2 parameters
# # lon0, lat0 = -85.5, 10
# # rot = 45.0

# # Define slice positions in Mesh 1's rotated coordinates
# rotx_CK_km = np.array([-50, -30, -10, 10, 30, 50])
# y_min_CK, y_max_CK = -100, 100

# # For each slice position in Mesh 1, find the corresponding position in Mesh 2
# rotx_km = []
# roty_ranges = []  # Store the y ranges for each slice

# for rotx_1 in rotx_CK_km:
#     # Step 1: Create points along the slice in Mesh 1 rotated coords
#     roty_1_array = np.linspace(y_min_CK, y_max_CK, 100)  # km
#     rotx_1_array = np.full_like(roty_1_array, rotx_1)  # constant rotx

#     # Step 2: Un-rotate to Mesh 1 local coords
#     x_local_1_km, y_local_1_km = utp.rot_xy(rotx_1_array, roty_1_array, rot=-rot_CK)

#     # Step 3: Add offsets to get Mesh 1 full coords (in meters)
#     x_full_1_m = x_local_1_km * 1e3 + x0_CK
#     y_full_1_m = y_local_1_km * 1e3 + y0_CK

#     # Step 4: Convert to lon/lat
#     lon_array, lat_array = utp.ckm2LLd(x_full_1_m, y_full_1_m, lon0_CK, lat0_CK, -rot_CK)

#     # Step 5: Convert to Mesh 2 local coords (km)
#     x_km, y_km = utp.azi_equidist_proj(lon_array, lat_array, lon0, lat0)

#     # Step 6: Rotate to Mesh 2 rotated coords
#     rotx_2_array, roty_2_array = utp.rot_xy(x_km, y_km, rot=rot)

#     # Step 7: Take mean of rotx values
#     rotx_2_mean = np.mean(rotx_2_array)
#     rotx_km.append(rotx_2_mean)

#     # Step 8: Get the y range in Mesh 2
#     roty_2_min = np.min(roty_2_array)
#     roty_2_max = np.max(roty_2_array)
#     roty_ranges.append((roty_2_min, roty_2_max))

#     print(f"Mesh 1 rotx={rotx_1:6.1f} km, roty=[{y_min_CK:6.1f}, {y_max_CK:6.1f}] km")
#     print(f"  → Mesh 2 rotx={rotx_2_mean:6.1f} km (std: {np.std(rotx_2_array):.3f}), "
#         f"roty=[{roty_2_min:6.1f}, {roty_2_max:6.1f}] km")

# rotx_km = np.array(rotx_km)
# roty_ranges = np.array(roty_ranges)

# # Determine common y range for Mesh 2 (conservative: intersection of all ranges)
# y_min = np.max(roty_ranges[:, 0])  # Maximum of all minimums
# y_max = np.min(roty_ranges[:, 1])  # Minimum of all maximums

# # Or liberal: union of all ranges
# y_min_liberal = np.min(roty_ranges[:, 0])
# y_max_liberal = np.max(roty_ranges[:, 1])

# print("\n=== Summary ===")
# print("Mesh 1 rotx (km):", rotx_CK_km)
# print("Mesh 2 rotx (km):", rotx_km)
# print("Difference (km):", rotx_km - rotx_CK_km)
# print(f"\nMesh 1 roty range: [{y_min_CK}, {y_max_CK}] km")
# print(f"Mesh 2 roty range (conservative): [{y_min:.1f}, {y_max:.1f}] km")
# print(f"Mesh 2 roty range (liberal): [{y_min_liberal:.1f}, {y_max_liberal:.1f}] km")

# # Use these for Mesh 2 slices
# x_strike_km = rotx_km
# x_strike_values = [x * 1e3 for x in x_strike_km]
# # y_min, y_max = y_min_liberal, y_max_liberal  # or use conservative

In [ ]:
# # Define along-dip slice positions in Mesh 1's rotated coordinates
# roty_dip_CK_km = np.arange(-50, 51, 20)  # [-50, -30, -10, 10, 30, 50]
# x_min_CK, x_max_CK = -100, 100  # km

# # For each along-dip slice position in Mesh 1, find the corresponding position in Mesh 2
# roty_dip_km = []
# rotx_dip_ranges = []  # Store the x ranges for each slice

# print("\n=== Converting Along-Dip Slices from Mesh 1 to Mesh 2 ===")
# for roty_1 in roty_dip_CK_km:
#     # Step 1: Create points along the slice in Mesh 1 rotated coords
#     rotx_1_array = np.linspace(x_min_CK, x_max_CK, 100)  # km, varying along-dip
#     roty_1_array = np.full_like(rotx_1_array, roty_1)  # constant roty

#     # Step 2: Un-rotate to Mesh 1 local coords
#     x_local_1_km, y_local_1_km = utp.rot_xy(rotx_1_array, roty_1_array, rot=-rot_CK)

#     # Step 3: Add offsets to get Mesh 1 full coords (in meters)
#     x_full_1_m = x_local_1_km * 1e3 + x0_CK
#     y_full_1_m = y_local_1_km * 1e3 + y0_CK

#     # Step 4: Convert to lon/lat
#     lon_array, lat_array = utp.ckm2LLd(x_full_1_m, y_full_1_m, lon0_CK, lat0_CK, -rot_CK)

#     # Step 5: Convert to Mesh 2 local coords (km)
#     x_km, y_km = utp.azi_equidist_proj(lon_array, lat_array, lon0, lat0)

#     # Step 6: Rotate to Mesh 2 rotated coords
#     rotx_2_array, roty_2_array = utp.rot_xy(x_km, y_km, rot=rot)

#     # Step 7: Take mean of roty values (should be ~constant)
#     roty_2_mean = np.mean(roty_2_array)
#     roty_dip_km.append(roty_2_mean)

#     # Step 8: Get the x range in Mesh 2
#     rotx_2_min = np.min(rotx_2_array)
#     rotx_2_max = np.max(rotx_2_array)
#     rotx_dip_ranges.append((rotx_2_min, rotx_2_max))

#     print(f"Mesh 1 roty={roty_1:6.1f} km, rotx=[{x_min_CK:6.1f}, {x_max_CK:6.1f}] km")
#     print(f"  → Mesh 2 roty={roty_2_mean:6.1f} km (std: {np.std(roty_2_array):.3f}), "
#         f"rotx=[{rotx_2_min:6.1f}, {rotx_2_max:6.1f}] km")

# roty_dip_km = np.array(roty_dip_km)
# rotx_dip_ranges = np.array(rotx_dip_ranges)

# # Determine common x range for Mesh 2 (conservative: intersection of all ranges)
# x_min = np.max(rotx_dip_ranges[:, 0])  # Maximum of all minimums
# x_max = np.min(rotx_dip_ranges[:, 1])  # Minimum of all maximums

# # Or liberal: union of all ranges
# x_min_liberal = np.min(rotx_dip_ranges[:, 0])
# x_max_liberal = np.max(rotx_dip_ranges[:, 1])

# print("\n=== Summary for Along-Dip Slices ===")
# print("Mesh 1 roty (km):", roty_dip_CK_km)
# print("Mesh 2 roty (km):", roty_dip_km)
# print("Difference (km):", roty_dip_km - roty_dip_CK_km)
# print(f"\nMesh 1 rotx range: [{x_min_CK}, {x_max_CK}] km")
# print(f"Mesh 2 rotx range (conservative): [{x_min:.1f}, {x_max:.1f}] km")
# print(f"Mesh 2 rotx range (liberal): [{x_min_liberal:.1f}, {x_max_liberal:.1f}] km")

# # Use these for Mesh 2 along-dip slices
# y_dip_km = roty_dip_km
# y_dip_values = [y * 1e3 for y in y_dip_km]
# # x_min, x_max are already defined above (use liberal or conservative)

In [ ]:
import numpy as np
from scipy.optimize import minimize

# Mesh 1 parameters
lon0_CK, lat0_CK = -84, 7
rot_CK = 45.0
x0_CK, y0_CK = 130e3, 350e3

# Mesh 2 parameters
lon0, lat0 = -85.5, 10
rot = 45.0

# Define slices in Mesh 1's rotated coordinates
rotx_CK_km = np.array([-50, -30, -10, 10, 30, 50])
y_min_CK, y_max_CK = -100, 100

# Step 1: Convert Mesh 1 slices to geographic paths (lon/lat)
print("=== Converting Mesh 1 slices to geographic coordinates ===")
geographic_paths_strike = []

for rotx_1 in rotx_CK_km:
    # Sample points along the slice in Mesh 1
    roty_1_array = np.linspace(y_min_CK, y_max_CK, 100)
    rotx_1_array = np.full_like(roty_1_array, rotx_1)

    # Un-rotate to Mesh 1 local coords
    x_local_1_km, y_local_1_km = utp.rot_xy(rotx_1_array, roty_1_array, rot=-rot_CK)

    # Add offsets
    x_full_1_m = x_local_1_km * 1e3 + x0_CK
    y_full_1_m = y_local_1_km * 1e3 + y0_CK

    # Convert to lon/lat (geographic coordinates)
    lon_path, lat_path = utp.ckm2LLd(x_full_1_m, y_full_1_m, lon0_CK, lat0_CK, -rot_CK)

    geographic_paths_strike.append((lon_path, lat_path))
    print(f"Slice rotx={rotx_1} km: lon=[{lon_path.min():.2f}, {lon_path.max():.2f}], "
        f"lat=[{lat_path.min():.2f}, {lat_path.max():.2f}]")

# Step 2: For Mesh 2, convert geographic paths to Mesh 2 coords and determine slice planes
print("\n=== Determining slice positions in Mesh 2 ===")
x_strike_km = []

for i, (lon_path, lat_path) in enumerate(geographic_paths_strike):
    # Convert geographic path to Mesh 2 local coords
    x_mesh2_km, y_mesh2_km = utp.azi_equidist_proj(lon_path, lat_path, lon0, lat0)

    # Rotate to Mesh 2 rotated coords
    rotx_2_array, roty_2_array = utp.rot_xy(x_mesh2_km, y_mesh2_km, rot=rot)

    # Take mean rotx as the slice position in Mesh 2
    rotx_2_mean = np.mean(rotx_2_array)
    x_strike_km.append(rotx_2_mean)

    print(f"Geographic path {i} → Mesh 2 rotx={rotx_2_mean:.2f} km (std={np.std(rotx_2_array):.2f})")

x_strike_km = np.array(x_strike_km)
x_strike_values = [x * 1e3 for x in x_strike_km]

# Determine y range for Mesh 2
# Use the geographic extent to determine appropriate y range
all_roty_2 = []
for lon_path, lat_path in geographic_paths_strike:
    x_mesh2_km, y_mesh2_km = utp.azi_equidist_proj(lon_path, lat_path, lon0, lat0)
    rotx_2, roty_2 = utp.rot_xy(x_mesh2_km, y_mesh2_km, rot=rot)
    all_roty_2.extend(roty_2)

y_min = np.min(all_roty_2)
y_max = np.max(all_roty_2)

print(f"\nMesh 2 along-strike slice positions (km): {x_strike_km}")
print(f"Mesh 2 roty range: [{y_min:.1f}, {y_max:.1f}] km")

# Do the same for along-dip slices
print("\n=== Converting Mesh 1 along-dip slices to geographic coordinates ===")
roty_dip_CK_km = np.arange(-50, 51, 20)
x_min_CK, x_max_CK = -100, 100

geographic_paths_dip = []

for roty_1 in roty_dip_CK_km:
    rotx_1_array = np.linspace(x_min_CK, x_max_CK, 100)
    roty_1_array = np.full_like(rotx_1_array, roty_1)

    x_local_1_km, y_local_1_km = utp.rot_xy(rotx_1_array, roty_1_array, rot=-rot_CK)
    x_full_1_m = x_local_1_km * 1e3 + x0_CK
    y_full_1_m = y_local_1_km * 1e3 + y0_CK
    lon_path, lat_path = utp.ckm2LLd(x_full_1_m, y_full_1_m, lon0_CK, lat0_CK, -rot_CK)

    geographic_paths_dip.append((lon_path, lat_path))
    print(f"Slice roty={roty_1} km: lon=[{lon_path.min():.2f}, {lon_path.max():.2f}], "
        f"lat=[{lat_path.min():.2f}, {lat_path.max():.2f}]")

print("\n=== Determining along-dip slice positions in Mesh 2 ===")
y_dip_km = []

for i, (lon_path, lat_path) in enumerate(geographic_paths_dip):
    x_mesh2_km, y_mesh2_km = utp.azi_equidist_proj(lon_path, lat_path, lon0, lat0)
    rotx_2_array, roty_2_array = utp.rot_xy(x_mesh2_km, y_mesh2_km, rot=rot)

    roty_2_mean = np.mean(roty_2_array)
    y_dip_km.append(roty_2_mean)

    print(f"Geographic path {i} → Mesh 2 roty={roty_2_mean:.2f} km (std={np.std(roty_2_array):.2f})")

y_dip_km = np.array(y_dip_km)
y_dip_values = [y * 1e3 for y in y_dip_km]

# Determine x range for Mesh 2
all_rotx_2 = []
for lon_path, lat_path in geographic_paths_dip:
    x_mesh2_km, y_mesh2_km = utp.azi_equidist_proj(lon_path, lat_path, lon0, lat0)
    rotx_2, roty_2 = utp.rot_xy(x_mesh2_km, y_mesh2_km, rot=rot)
    all_rotx_2.extend(rotx_2)

x_min = np.min(all_rotx_2)
x_max = np.max(all_rotx_2)

print(f"\nMesh 2 along-dip slice positions (km): {y_dip_km}")
print(f"Mesh 2 rotx range: [{x_min:.1f}, {x_max:.1f}] km")

print("\n=== Final Summary ===")
print("Along-strike slices:")
print(f"  Mesh 1 positions: {rotx_CK_km}")
print(f"  Mesh 2 positions: {x_strike_km}")
print(f"  Mesh 2 y range: [{y_min:.1f}, {y_max:.1f}] km")
print("\nAlong-dip slices:")
print(f"  Mesh 1 positions: {roty_dip_CK_km}")
print(f"  Mesh 2 positions: {y_dip_km}")
print(f"  Mesh 2 x range: [{x_min:.1f}, {x_max:.1f}] km")

In [ ]:
# IMPROVED VERSION: Horizontal slices using PyVista native slicing
# This gives smoother results than direct griddata interpolation

depth_levels = np.arange(0.0, -80.1, -10.0)  # 0 to -80 km, 9 depths
lon_spacing = 0.01
lat_spacing = 0.01

# Create grid for interpolation
lon_vals = np.arange(region_fault[0], region_fault[1] + lon_spacing, lon_spacing)
lat_vals = np.arange(region_fault[2], region_fault[3] + lat_spacing, lat_spacing)
lon_grid, lat_grid = np.meshgrid(lon_vals, lat_vals)
grid_shape = (len(lat_vals), len(lon_vals))

# Convert to mesh coordinates (NO rotation - mesh aligns with N-E)
x_mesh_km, y_mesh_km = utp.azi_equidist_proj(lon_grid.ravel(), lat_grid.ravel(), lon0, lat0)

# Convert from km to meters for PyVista operations
x_mesh_m = x_mesh_km * 1e3
y_mesh_m = y_mesh_km * 1e3

mu_slice_list_pv = []
anom_slice_list_pv = []

print("Creating horizontal slices using PyVista slicing...")
for i, depth_km in enumerate(depth_levels):
    z_value = depth_km * 1e3  # Convert to meters

    # Use PyVista's native slicing
    normal = [0., 0., 1.]  # Horizontal slice
    origin = [0., 0., z_value]

    # Slice the 3D and homogeneous grids
    slice_3d = mu_3d_grid.slice(normal=normal, origin=origin)
    slice_hom = mu_hom_grid.slice(normal=normal, origin=origin)

    if len(slice_3d.points) == 0:
        print(f"  Depth {depth_km:.0f} km: No data")
        mu_slice_list_pv.append(np.full(grid_shape, np.nan))
        anom_slice_list_pv.append(np.full(grid_shape, np.nan))
        continue

    # Interpolate onto our regular grid
    from scipy.interpolate import griddata
    mu_interp = griddata(slice_3d.points[:, :2], slice_3d['shear modulus'],
                        (x_mesh_m.reshape(grid_shape), y_mesh_m.reshape(grid_shape)),
                        method='linear')
    mu_hom_interp = griddata(slice_hom.points[:, :2], slice_hom['shear modulus'],
                            (x_mesh_m.reshape(grid_shape), y_mesh_m.reshape(grid_shape)),
                            method='linear')

    # Calculate anomaly
    mu_anom = (mu_interp - mu_ref) / mu_ref * 100

    mu_slice_list_pv.append(mu_interp)
    anom_slice_list_pv.append(mu_anom)

    n_valid = np.sum(~np.isnan(mu_interp))
    print(f"  Depth {depth_km:.0f} km: {n_valid}/{grid_shape[0]*grid_shape[1]} valid points")

# Compute color scale ranges
mu_slice_min_pv = float(np.nanmin([np.nanmin(slc) for slc in mu_slice_list_pv]))
mu_slice_max_pv = float(np.nanmax([np.nanmax(slc) for slc in mu_slice_list_pv]))
mu_slice_step_pv = max((mu_slice_max_pv - mu_slice_min_pv) / 20.0, 1e-3)

anom_slice_vmax_pv = float(np.nanmax([np.nanmax(np.abs(slc)) for slc in
anom_slice_list_pv]))
anom_slice_step_pv = max(anom_slice_vmax_pv / 10.0, 1e-3)

print(f"\nμ range: [{mu_slice_min_pv:.1f}, {mu_slice_max_pv:.1f}] GPa, step: {mu_slice_step_pv:.4f} GPa")
print(f"Using μ_ref = {mu_ref:.1f} GPa for anomaly calculation")
print(f"Anomaly range: [{-anom_slice_vmax_pv:.1f}, {anom_slice_vmax_pv:.1f}] %, step: {anom_slice_step_pv:.4f} %")

# Define vertical slice positions IN ROTATED COORDINATES
# After rotating 45° CCW: rotx is along-dip (NE), roty is along-strike (NW-SE)
rot = 45.0  # degrees

# Along-strike slices: constant rotx (along-dip), varying roty (along-strike)
# x_strike_km = np.arange(-50, 51, 20)  # Rotated-x positions (along-dip direction)
x_strike_km = np.arange(-39, 62, 20)  # Rotated-x positions (along-dip direction)
y_min, y_max = -100, 100              # Rotated-y range (along-strike extent)

# Along-dip slices: constant roty (along-strike), varying rotx (along-dip)  
# y_dip_km = np.arange(-50, 51, 20)     # Rotated-y positions (along-strike direction)
y_dip_km = np.arange(-53, 51, 20)     # Rotated-y positions (along-strike direction)
x_min, x_max = -100, 100              # Rotated-x range (along-dip extent)

if het3d_str == '_DeShon3D_ref_4':
    # series_mu = [mu_slice_min_pv, mu_slice_max_pv, mu_slice_step_pv]
    # series_mu = [5, 155, (155 - 5) / 20.0]
    series_mu = [5, 180, (180 - 5) / 20.0]

    # series_mu_ano = [-anom_slice_vmax_pv, anom_slice_vmax_pv, anom_slice_step_pv]
    # series_mu_ano = [-100, 100, 100/10.0]
    # series_mu_ano = [-150, 150, 140/10.0]
    series_mu_ano = [-160, 160, 160/10.0]

elif het3d_str == '_mul55u25':
    series_mu = [mu_slice_min_pv, mu_slice_max_pv, mu_slice_step_pv]
    # series_mu = [25, 55, (55 - 25) / 20.0]
    series_mu_ano = [-anom_slice_vmax_pv, anom_slice_vmax_pv, anom_slice_step_pv]

# Plot PyVista-sliced data using the same plotting function
# The function will convert rotated slice positions back to lon/lat for plotting

# Plot μ slices (PyVista version)
mu_slice_file_pv = resultpath + 'mu_horizontal_slices_pyvista_slab2.png'
fig_mu_pv = _plot_slice_panels(mu_slice_list_pv, depth_levels, cmap="rainbow",
                                series=series_mu,
                                label="μ (GPa)", filename=mu_slice_file_pv)

# Plot anomaly slices (PyVista version)
anom_slice_file_pv = resultpath + 'mu_anomaly_horizontal_slices_pyvista.png'
fig_anom_pv = _plot_slice_panels(anom_slice_list_pv, depth_levels, cmap="roma",
                                series=series_mu_ano,
                                label="μ anomaly (%)", filename=anom_slice_file_pv)

print("\nPyVista-based slices complete!")

## 10.6. Vertical Slices Along Strike and Dip

Visualize μ structure in vertical cross-sections to understand how structure varies with depth along different directions.

**Slice Orientations:**
- **Along-strike slices**: Constant x (dip direction), varying y (strike) and z (depth)
- **Along-dip slices**: Constant y (strike direction), varying x (dip) and z (depth)

**Key Features:**
- Cartesian projection (x-z or y-z) in PyGMT
- Plate interface (fault) overlaid as reference line
- Shows where slip occurs relative to μ structure
- Multiple slice positions to capture spatial variability

**Implementation:**
- Use PyVista slicing along vertical planes
- Interpolate plate interface onto slice coordinates
- Plot in PyGMT with simple X-axis (distance) vs Z-axis (depth)

Visualize μ structure on vertical cross-sections along strike and dip directions,
with plate interface overlay for geological context.


In [ ]:
# Prepare plate interface in rotated coordinates for vertical slicing
# Note: df_plate has columns ['lon', 'lat', 'z'] where z is depth in km
df_plate_interface = df_plate.copy()

# Convert lon/lat to mesh coordinates (km) aligned with N-E
df_plate_interface['x'], df_plate_interface['y'] = utp.azi_equidist_proj(
    df_plate_interface['lon'], df_plate_interface['lat'], lon0, lat0
)
df_plate_interface['z'] = df_plate_interface['dep']

# Get mesh coords in km
x_mesh_km = df_plate_interface['x'].values
y_mesh_km = df_plate_interface['y'].values

# Apply rotation to convert mesh coords to rotated coords for vertical slicing
# Rotate by 45° CCW so rotx = along-dip (NE), roty = along-strike (NW-SE)
x_rot_km, y_rot_km = utp.rot_xy(x_mesh_km, y_mesh_km, rot=rot)

# Store in dataframe (in meters for PyVista operations)
df_plate_interface['x_rot'] = x_rot_km * 1e3  # rotated x in meters (along-dip)
df_plate_interface['y_rot'] = y_rot_km * 1e3  # rotated y in meters (along-strike)
df_plate_interface['z_rot'] = df_plate_interface['z'].values * 1e3  # z in meters

print(f"Plate interface: {len(df_plate_interface)} points")
print(f"X_rot range (along-dip): [{df_plate_interface['x_rot'].min()/1e3:.1f}, {df_plate_interface['x_rot'].max()/1e3:.1f}] km")
print(f"Y_rot range (along-strike): [{df_plate_interface['y_rot'].min()/1e3:.1f}, {df_plate_interface['y_rot'].max()/1e3:.1f}] km")
print(f"Z range: [{df_plate_interface['z_rot'].min()/1e3:.1f}, {df_plate_interface['z_rot'].max()/1e3:.1f}] km")

In [ ]:
# def extract_vertical_slice_pygmt(mu_grid, normal, origin, slice_axis='y-z',
#                                    field_name='shear modulus'):
#     """
#     Extract vertical slice from PyVista grid and prepare for PyGMT plotting.

#     Parameters:
#     -----------
#     mu_grid : pyvista.UnstructuredGrid
#         The 3D grid with mu values
#     normal : list
#         Normal vector for the slice plane [nx, ny, nz]
#     origin : list
#         Origin point for the slice plane [x, y, z]
#     slice_axis : str
#         Which axes to plot: 'y-z' for along-strike, 'x-z' for along-dip
#     field_name : str
#         Name of the field to extract from the grid

#     Returns:
#     --------
#     slice_df : pd.DataFrame
#         DataFrame with columns [coord1, z, mu] ready for PyGMT
#     """
#     # Slice the grid
#     slice_obj = mu_grid.slice(normal=normal, origin=origin)

#     if len(slice_obj.points) == 0:
#         print(f"Warning: Empty slice at origin {origin}")
#         return None

#     # Extract coordinates and mu values
#     points = slice_obj.points  # [N, 3] array
#     mu_vals = slice_obj[field_name]

#     # Select appropriate coordinates based on slice direction
#     if slice_axis == 'y-z':
#         # Along-strike slice (constant x)
#         coord1 = points[:, 1] / 1e3  # y in km
#         coord1_name = 'y_km'
#     elif slice_axis == 'x-z':
#         # Along-dip slice (constant y)
#         coord1 = points[:, 0] / 1e3  # x in km
#         coord1_name = 'x_km'
#     else:
#         raise ValueError(f"Unknown slice_axis: {slice_axis}")

#     z_km = points[:, 2] / 1e3  # z in km

#     # Create DataFrame
#     slice_df = pd.DataFrame({
#         coord1_name: coord1,
#         'z_km': z_km,
#         'mu': mu_vals
#     })

#     return slice_df


In [ ]:
# Ensure mu_3d_grid and mu_hom_grid are PyVista objects
# If they were overwritten, reload them

if not hasattr(mu_3d_grid, 'slice'):
    print("Reloading mu_3d_grid as PyVista object...")
    mu_3d_grid = load_xdmf_as_pyvista(mu_3d_file)

if not hasattr(mu_hom_grid, 'slice'):
    print("Reloading mu_hom_grid as PyVista object...")
    mu_hom_grid = load_xdmf_as_pyvista(mu_hom_file)

print(f"mu_3d_grid type: {type(mu_3d_grid)}")
print(f"mu_hom_grid type: {type(mu_hom_grid)}")
print("Ready for slicing!")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from scipy.interpolate import griddata
import numpy as np

# # Define x positions for along-strike slices in ROTATED coordinates
# x_strike_km = np.arange(-50, 51, 20)  # Rotated-x positions (along-dip)
x_strike_km = np.arange(-39, 62, 20)  # Rotated-x positions (along-dip direction)
x_strike_values = [x * 1e3 for x in x_strike_km]  # Convert to meters

# Choose what to plot: 'mu' or 'anomaly'
plot_type = 'mu'  # Options: 'mu' or 'anomaly'
# plot_type = 'anomaly'  # Options: 'mu' or 'anomaly'

# Use df_plate_interface which already has rotated coordinates
df_plate_for_slicing = df_plate_interface.copy()

print(f"Plate interface rotated coordinates:")
print(f"  X_rot: [{df_plate_for_slicing['x_rot'].min()/1e3:.1f}, \
{df_plate_for_slicing['x_rot'].max()/1e3:.1f}] km")
print(f"  Y_rot: [{df_plate_for_slicing['y_rot'].min()/1e3:.1f}, \
{df_plate_for_slicing['y_rot'].max()/1e3:.1f}] km")

# Prepare 2D interpolation for plate interface in rotated coords
plate_xy_rot = np.column_stack([
    df_plate_for_slicing['x_rot'].values,
    df_plate_for_slicing['y_rot'].values
])
plate_z = df_plate_for_slicing['z_rot'].values

# Create matplotlib figure
fig, axes = plt.subplots(3, 2, figsize=(7, 5), dpi=300, sharex=True, sharey=True)
axes = axes.flatten()

# Define region (roty-z space in rotated coordinates)
y_min, y_max = -100, 100  # km
z_min, z_max = -80, 0     # km

# Create regular grid in roty-z space
n_y_points = 200
n_z_points = 160
y_grid_m = np.linspace(y_min * 1e3, y_max * 1e3, n_y_points)
z_grid_m = np.linspace(z_min * 1e3, z_max * 1e3, n_z_points)
yy_rot, zz = np.meshgrid(y_grid_m, z_grid_m)

# Normal for along-strike slice (constant rotx plane)
# In mesh coords: rotx = x*cos(45) + y*sin(45) = constant
# So: x + y = constant, normal = [1, 1, 0]
normal = [1., 1., 0.]

# Collect all values for color limits
all_values = []
slice_data = []

print("Creating slices...")
for i, (x_km, rotx_val) in enumerate(zip(x_strike_km, x_strike_values)):
    # Origin: a point where rotx = rotx_val
    # rotx = x*cos(45) + y*sin(45), choose x = y = rotx_val/sqrt(2)
    origin_coord = rotx_val / np.sqrt(2)
    origin = [origin_coord, origin_coord, 0.]

    # Slice both grids
    slice_3d = mu_3d_grid.slice(normal=normal, origin=origin)
    slice_hom = mu_hom_grid.slice(normal=normal, origin=origin)

    if len(slice_3d.points) == 0:
        print(f"  Slice at rotx = {x_km} km: No data")
        slice_data.append(None)
        continue

    # Get slice points in mesh coordinates and rotate to rotated coords
    points_mesh = slice_3d.points  # meters
    x_rot_3d, y_rot_3d = utp.rot_xy(points_mesh[:, 0] / 1e3, points_mesh[:, 1] / 1e3, rot=rot)
    z_3d = points_mesh[:, 2] / 1e3

    points_hom = slice_hom.points
    x_rot_hom, y_rot_hom = utp.rot_xy(points_hom[:, 0] / 1e3, points_hom[:, 1] / 1e3, rot=rot)
    z_hom = points_hom[:, 2] / 1e3

    # Get mu values
    mu_3d_slice = slice_3d['shear modulus']
    mu_hom_slice = slice_hom['shear modulus']

    # Interpolate BOTH to same regular grid in rotated roty-z space
    mu_3d_interp = griddata(
        np.column_stack([y_rot_3d * 1e3, z_3d * 1e3]),
        mu_3d_slice,
        (yy_rot, zz),
        method='linear'
    )

    mu_hom_interp = griddata(
        np.column_stack([y_rot_hom * 1e3, z_hom * 1e3]),
        mu_hom_slice,
        (yy_rot, zz),
        method='linear'
    )

    # Choose what to plot
    if plot_type == 'anomaly':
        plot_values = (mu_3d_interp - mu_ref) / mu_ref * 100
        cmap = 'cmc.roma'
        cbar_label = 'μ Anomaly (%)'
    else:
        plot_values = mu_3d_interp
        cmap = 'gist_rainbow_r'
        cbar_label = 'Shear Modulus μ (GPa)'

    slice_data.append(plot_values)
    all_values.append(plot_values[~np.isnan(plot_values)])

    n_valid = np.sum(~np.isnan(plot_values))
    print(f"  Slice at rotx = {x_km} km: {n_valid} valid points")

# Check if we have any data
if len(all_values) == 0:
    raise ValueError("No valid slices found! Check that x_strike_km values are within the model domain.")

# Determine color limits
all_vals = np.concatenate(all_values)
if plot_type == 'anomaly':
    vmax = np.abs(all_vals).max()
    if het3d_str == '_DeShon3D_ref_4':
        # vmax = (155 - mu_ref) / mu_ref * 100
        vmax = 150  # Cap at 150% for better color contrast
        vmax = 160  # Cap at 160% for better color contrast
    vmin = -vmax
    print(f"Anomaly range: [{all_vals.min():.1f}, {all_vals.max():.1f}]% (using symmetric±{vmax:.1f}%)")
else:
    vmin = all_vals.min()
    vmax = all_vals.max()
    if het3d_str == '_DeShon3D_ref_4':
        vmax = 155  # Cap at 155 for better color contrast
        vmax = 180  # Cap at 155 for better color contrast
    print(f"μ range: [{all_vals.min():.1f}, {all_vals.max():.1f}] GPa (using {vmax:.1f} GPa)")

# Now plot
for i, (x_km, plot_values) in enumerate(zip(x_strike_km, slice_data)):
    ax = axes[i]

    if plot_values is None:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                transform=ax.transAxes, fontsize=14)
        continue

    if plot_type == 'anomaly':
        cmap = 'cmc.roma'
    else:
        cmap = 'gist_rainbow_r'

    # Convert to km for plotting
    yy_km = yy_rot / 1e3
    zz_km = zz / 1e3

    contour_levels = np.linspace(vmin, vmax, 20)
    cs = ax.contourf(yy_km, zz_km, plot_values, levels=contour_levels, cmap=cmap,
                    vmin=vmin, vmax=vmax, extend='both')

    # Add depth level lines
    for depth in depth_levels:
        ax.axhline(y=depth, color='white', linestyle='--', linewidth=0.5, zorder=5)

    # Plot plate interface
    rotx_val = x_strike_values[i]
    y_interface_m = np.linspace(y_min * 1e3, y_max * 1e3, 200)
    x_interface_m = np.full_like(y_interface_m, rotx_val)

    z_interface = griddata(
        plate_xy_rot,
        plate_z,
        np.column_stack([x_interface_m, y_interface_m]),
        method='linear'
    )

    valid = ~np.isnan(z_interface)
    if valid.sum() > 0:
        interface_y_km = y_interface_m[valid] / 1e3
        interface_z_km = z_interface[valid] / 1e3
        ax.plot(interface_y_km, interface_z_km, 'k-', linewidth=1.5, zorder=10)

    ax.set_xlim(y_min, y_max)
    ax.set_ylim(z_min, z_max)
    ax.set_aspect('equal')
    ax.set_title(f'x = {x_km} km', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, linewidth=0.5)

    if i % 2 == 0:
        ax.set_ylabel('Depth (km)', fontsize=11)
    if i >= 4:
        ax.set_xlabel('Along-strike (km)', fontsize=11)

# Colorbar
plt.tight_layout(rect=[0, 0.05, 1, 1])
cbar_ax = fig.add_axes([0.15, 0.01, 0.7, 0.02])
cbar = fig.colorbar(cs, cax=cbar_ax, orientation='horizontal')
cbar.set_label(cbar_label, fontsize=12)
if het3d_str == '_DeShon3D_ref_4':
    if plot_type == 'mu':
        cbar.set_ticks(np.arange(vmin, vmax+1, 20))  # 5, 25, 45, 65, 85, 105, 125, 145
    else:
        cbar.set_ticks(np.arange(-150, 150+1, 50))
elif het3d_str == '_mul55u25':
    if plot_type == 'mu':
        cbar.set_ticks(np.arange(25, 55+1, 5))  # 5, 25, 45, 65, 85, 105, 125, 145
    else:
        cbar.set_ticks(np.arange(-40, 40+1, 10))

output_file = resultpath + f'mu_vertical_slices_strike_{plot_type}.png'
plt.savefig(output_file, dpi=300, bbox_inches='tight')
print(f"\nAlong-strike slices saved to: {output_file}")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from scipy.interpolate import griddata
import numpy as np

# Define y positions for along-dip slices in ROTATED coordinates
# y_dip_km = np.arange(-50, 51, 20)  # Rotated-y positions (along-strike direction)
y_dip_km = np.arange(-53, 51, 20)     # Rotated-y positions (along-strike direction)
y_dip_values = [y * 1e3 for y in y_dip_km]  # Convert to meters

# Choose what to plot: 'mu' or 'anomaly'
plot_type = 'mu'  # Options: 'mu' or 'anomaly'
# plot_type = 'anomaly'  # Options: 'mu' or 'anomaly'

# Use df_plate_interface which already has rotated coordinates
df_plate_for_slicing = df_plate_interface.copy()

print(f"Plate interface rotated coordinates:")
print(f"  X_rot: [{df_plate_for_slicing['x_rot'].min()/1e3:.1f}, \
{df_plate_for_slicing['x_rot'].max()/1e3:.1f}] km")
print(f"  Y_rot: [{df_plate_for_slicing['y_rot'].min()/1e3:.1f}, \
{df_plate_for_slicing['y_rot'].max()/1e3:.1f}] km")

# Prepare 2D interpolation for plate interface in rotated coords
plate_xy_rot = np.column_stack([
    df_plate_for_slicing['x_rot'].values,
    df_plate_for_slicing['y_rot'].values
])
plate_z = df_plate_for_slicing['z_rot'].values

# Create matplotlib figure
fig, axes = plt.subplots(3, 2, figsize=(7, 5), dpi=300, sharex=True, sharey=True)
axes = axes.flatten()

# Define region (rotx-z space in rotated coordinates)
x_min, x_max = -100, 100  # km (along-dip extent)
z_min, z_max = -80, 0     # km (depth)

# Create regular grid in rotx-z space
n_x_points = 200
n_z_points = 160
x_grid_m = np.linspace(x_min * 1e3, x_max * 1e3, n_x_points)
z_grid_m = np.linspace(z_min * 1e3, z_max * 1e3, n_z_points)
xx_rot, zz = np.meshgrid(x_grid_m, z_grid_m)

# Normal for along-dip slice (constant roty plane)
# In mesh coords: roty = -x*sin(45) + y*cos(45) = constant
# So: -x + y = constant, normal = [-1, 1, 0]
normal = [-1., 1., 0.]

# Collect all values for color limits
all_values = []
slice_data = []

print("Creating slices...")
for i, (y_km, roty_val) in enumerate(zip(y_dip_km, y_dip_values)):
    # Origin: a point where roty = roty_val
    # roty = -x*sin(45) + y*cos(45), choose x = -roty_val/sqrt(2), y = roty_val/sqrt(2)
    origin_x = -roty_val / np.sqrt(2)
    origin_y = roty_val / np.sqrt(2)
    origin = [origin_x, origin_y, 0.]

    # Slice both grids
    slice_3d = mu_3d_grid.slice(normal=normal, origin=origin)
    slice_hom = mu_hom_grid.slice(normal=normal, origin=origin)

    if len(slice_3d.points) == 0:
        print(f"  Slice at roty = {y_km} km: No data")
        slice_data.append(None)
        continue

    # Get slice points in mesh coordinates and rotate to rotated coords
    points_mesh = slice_3d.points  # meters
    x_rot_3d, y_rot_3d = utp.rot_xy(points_mesh[:, 0] / 1e3, points_mesh[:, 1] / 1e3, rot=rot)
    z_3d = points_mesh[:, 2] / 1e3

    points_hom = slice_hom.points
    x_rot_hom, y_rot_hom = utp.rot_xy(points_hom[:, 0] / 1e3, points_hom[:, 1] / 1e3, rot=rot)
    z_hom = points_hom[:, 2] / 1e3

    # Get mu values
    mu_3d_slice = slice_3d['shear modulus']
    mu_hom_slice = slice_hom['shear modulus']

    # Interpolate BOTH to same regular grid in rotated rotx-z space
    mu_3d_interp = griddata(
        np.column_stack([x_rot_3d * 1e3, z_3d * 1e3]),
        mu_3d_slice,
        (xx_rot, zz),
        method='linear'
    )

    mu_hom_interp = griddata(
        np.column_stack([x_rot_hom * 1e3, z_hom * 1e3]),
        mu_hom_slice,
        (xx_rot, zz),
        method='linear'
    )

    # Choose what to plot
    if plot_type == 'anomaly':
        plot_values = (mu_3d_interp - mu_ref) / mu_ref * 100
        cmap = 'cmc.roma'
        cbar_label = 'μ Anomaly (%)'
    else:
        plot_values = mu_3d_interp
        cmap = 'gist_rainbow_r'
        cbar_label = 'Shear Modulus μ (GPa)'

    slice_data.append(plot_values)
    all_values.append(plot_values[~np.isnan(plot_values)])

    n_valid = np.sum(~np.isnan(plot_values))
    print(f"  Slice at roty = {y_km} km: {n_valid} valid points")

# Check if we have any data
if len(all_values) == 0:
    raise ValueError("No valid slices found! Check that y_dip_km values are within the model domain.")

# Determine color limits
all_vals = np.concatenate(all_values)
if plot_type == 'anomaly':
    # Symmetric range for diverging colormap
    vmax = np.abs(all_vals).max()
    if het3d_str == '_DeShon3D_ref_4':
        # vmax = (155 - mu_ref) / mu_ref * 100
        vmax = 150  # Cap at 150% for better color contrast
        vmax = 160  # Cap at 160% for better color contrast
    vmin = -vmax
    print(f"Anomaly range: [{all_vals.min():.1f}, {all_vals.max():.1f}]% (using symmetric±{vmax:.1f}%)")
else:
    vmin = all_vals.min()
    vmax = all_vals.max()
    if het3d_str == '_DeShon3D_ref_4':
        vmax = 155  # Cap at 155 for better color contrast
        vmax = 180  # Cap at 155 for better color contrast
    print(f"μ range: [{all_vals.min():.1f}, {all_vals.max():.1f}] GPa (using {vmax:.1f} GPa)")

# Now plot
for i, (y_km, plot_values) in enumerate(zip(y_dip_km, slice_data)):
    ax = axes[i]

    if plot_values is None:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                transform=ax.transAxes, fontsize=14)
        continue

    if plot_type == 'anomaly':
        cmap = 'cmc.roma'
    else:
        cmap = 'gist_rainbow_r'

    # Convert to km for plotting
    xx_km = xx_rot / 1e3
    zz_km = zz / 1e3

    contour_levels = np.linspace(vmin, vmax, 20)
    cs = ax.contourf(xx_km, zz_km, plot_values, levels=contour_levels, cmap=cmap,
                    vmin=vmin, vmax=vmax, extend='both')

    # Add depth level lines
    for depth in depth_levels:
        ax.axhline(y=depth, color='white', linestyle='--', linewidth=0.5, zorder=5)

    # Plot plate interface
    roty_val = y_dip_values[i]
    x_interface_m = np.linspace(x_min * 1e3, x_max * 1e3, 200)
    y_interface_m = np.full_like(x_interface_m, roty_val)

    z_interface = griddata(
        plate_xy_rot,
        plate_z,
        np.column_stack([x_interface_m, y_interface_m]),
        method='linear'
    )

    valid = ~np.isnan(z_interface)
    if valid.sum() > 0:
        interface_x_km = x_interface_m[valid] / 1e3
        interface_z_km = z_interface[valid] / 1e3
        ax.plot(interface_x_km, interface_z_km, 'k-', linewidth=1.5, zorder=10)

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(z_min, z_max)
    ax.set_aspect('equal')
    ax.set_title(f'y = {y_km} km', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, linewidth=0.5)

    if i % 2 == 0:
        ax.set_ylabel('Depth (km)', fontsize=11)
    if i >= 4:
        ax.set_xlabel('Along-dip (km)', fontsize=11)

# Colorbar
plt.tight_layout(rect=[0, 0.05, 1, 1])
cbar_ax = fig.add_axes([0.15, 0.01, 0.7, 0.02])
cbar = fig.colorbar(cs, cax=cbar_ax, orientation='horizontal')
cbar.set_label(cbar_label, fontsize=12)
if het3d_str == '_DeShon3D_ref_4':
    if plot_type == 'mu':
        cbar.set_ticks(np.arange(vmin, vmax+1, 20))  # 5, 25, 45, 65, 85, 105, 125, 145
    else:
        cbar.set_ticks(np.arange(-150, 150+1, 50))
elif het3d_str == '_mul55u25':
    if plot_type == 'mu':
        cbar.set_ticks(np.arange(25, 55+1, 5))  # 5, 25, 45, 65, 85, 105, 125, 145
    else:
        cbar.set_ticks(np.arange(-40, 40+1, 10))

output_file = resultpath + f'mu_vertical_slices_dip_{plot_type}.png'
plt.savefig(output_file, dpi=300, bbox_inches='tight')
print(f"\nAlong-dip slices saved to: {output_file}")
plt.show()